In [ ]:
import tensorflow as tf
from keras.datasets import mnist
import cv2
import os
import pathlib
from keras.layers import Conv2D, Conv2DTranspose, Dropout, Dense, Reshape, LayerNormalization, LeakyReLU
from keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score, recall_score, precision_score

- The following study presents a model for generating chest X-ray images of normal subjects (without lung disease) and pneumonia patients.
- Through the proposed model, I tried to avoid most of the problems that the GAN models suffer from, in terms of the difficulty of training each of the generator and the discriminant, in addition to the problem of the modal collapse and the perceptual quality, so that I tried through the proposed model, to try to continue the training (to ensure the continuity of the derivability of cost function ) and discovering the features by the discriminator (the most accurate features for each case of the dataset), which leads the generator to focus on them during the training process.
- A conditional model was used for the GAN, and the discriminator was forced to determine whether the medical images are real or not, in addition to identifying the pathological condition in the generated images.
- I used (64, 64, 3) images because I didn't have enough computational resources.
- I used Google Colab For Training.
- Reading the images included in the dataset, which is for the sound health condition, and the other case, which is pneumonia.
- I have included all medical images included in each class, although the number of samples per class varies (thus this would require training for a higher number of Epochs for the GAN).

In [ ]:
class ReadDataset:
    def __init__(self, datasetpath, labels, image_shape):
        self.datasetpath = datasetpath
        self.labels = labels
        self.image_shape = image_shape
    def returListImages(self,):
        self.images = []
        for label in self.labels:
            self.images.append(list(pathlib.Path(os.path.join(self.datasetpath,
                                                              label)).glob('*.*')))
    def readImages(self,):
        self.returListImages()
        self.finalImages = []
        labels = []
        for label in range(len(self.labels)):
            for img in self.images[label]:
                img = cv2.imread(str(img))
                img = cv2.resize(img , self.image_shape)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img  = img/255
                self.finalImages.append(img)
                labels.append(label)
        images = np.array(self.finalImages)
        labels = np.array(labels)
        return images, labels

In [ ]:
readDatasetObject = ReadDataset('/input/chest-xray-pneumonia/chest_xray/train',
                               ['NORMAL', 'PNEUMONIA'],
                               (128, 128))
images, labels = readDatasetObject.readImages()

readDatasetObject = ReadDataset('/input/chest-xray-pneumonia/chest_xray/test',
                               ['NORMAL', 'PNEUMONIA'],
                               (128, 128))
images_test, labels_test = readDatasetObject.readImages()
images.shape, labels.shape, images_test.shape, labels_test.shape

In [ ]:
images.shape, labels.shape, images_test.shape, labels_test.shape

Sample images included in the dataset for each class

In [ ]:
plt.figure(figsize = (12, 12))
indexs = np.random.randint(0, len(labels), size = (64, ))
for i in range(64):
    plt.subplot(8, 8, (i + 1))
    plt.imshow(images[indexs[i]])
    plt.title(labels[indexs[i]])
plt.legend()

- The proposed generative adversarial network, so that the difficulty of training and many of the problems that can be encountered in the generative adversarial network can be avoided, such as postural collapse and cognitive quality.

- The structure focused on including many structures that helped some of them to avoid falling into the problem of situational collapse, and the other structures focused on perceptual quality (as it forced the distinguished network to focus more on the deeper features in the medical images, which helped the generator to capture them in the generation process).

- The architecture helped make the training balanced between both the Generator and the Discriminator.

- Conditional generation was used, whereby the Discriminator was forced to verify that the generated images were real, class-following (a healthy person, or a person with pneumonia).

- The generator's input was a noise with a regular distribution, in addition to the pathological condition that we want to classify.
- Use the MSE loss function to address the problem of cognitive quality and make the generator focus on the health characteristics of a healthy person, and the pathological characteristics of a person suffering from pneumonia.

In [ ]:
import tensorflow as tf
def jensen_shannon_divergence(y_true, y_pred):
    # Ensure probabilities sum to 1
    y_true = tf.keras.backend.clip(y_true, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())
    y_pred = tf.keras.backend.clip(y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())

    # Calculate average probabilities
    avg_prob = (y_true + y_pred) / 2

    # Calculate Jensen-Shannon Divergence
    divergence = 0.5 * tf.keras.backend.sum(y_true * tf.keras.backend.log(y_true / avg_prob))
    divergence += 0.5 * tf.keras.backend.sum(y_pred * tf.keras.backend.log(y_pred / avg_prob))

    return divergence
import tensorflow as tf
def wasserstein_loss(y_true, y_pred):
    return tf.reduce_mean(y_true * y_pred)

def gradient_penalty_loss(y_true, y_pred, interpolated_samples):
    gradients = tf.gradients(y_pred, interpolated_samples)[0]
    gradients_sqr = tf.square(gradients)
    gradients_sqr_sum = tf.reduce_sum(gradients_sqr, axis=list(range(1, len(gradients_sqr.shape))))

    gradient_penalty = tf.sqrt(gradients_sqr_sum)
    return tf.reduce_mean(tf.square(1 - gradient_penalty))

def combined_loss(y_true, y_pred):
    # Adjust weights as needed
    mse_loss = tf.keras.losses.MeanSquaredError()(y_true, y_pred)
    kl_loss = tf.keras.losses.KLDivergence()(y_true, y_pred)
    #js_loss = jensen_shannon_divergence(y_true, y_pred)
    w_loss = wasserstein_loss(y_true, y_pred)

    # Generate interpolated samples for WGAN-GP
    #alpha = tf.random.uniform(shape=[tf.shape(y_true)[0], 1, 1, 1], minval=0.0, maxval=1.0)
    #interpolated_samples = alpha * y_true + (1 - alpha) * y_pred

    # Add Wasserstein loss and gradient penalty
    #w_loss = tf.reduce_mean(y_true) - tf.reduce_mean(y_pred)
    #gp_loss = gradient_penalty_loss(y_true, y_pred, interpolated_samples)

    # Adjust weights for Wasserstein loss and gradient penalty
    return 0.89 * mse_loss + 0.01 * kl_loss + 0.1 * w_loss

def CBAM_Channel(output,reduction:int=16):
    #Channel_Attention
    #GlobalMaxPooling of input tensor
    x_global_max=layers.GlobalMaxPooling2D()(output)
    #GlobalAveragePooling of input tensor
    x_global_average=layers.GlobalAveragePooling2D()(output)
    #Feature_Reduction_Layer1
    reduced_feature=output.shape[3]//reduction
    x_dense_global_max_red= layers.Dense(reduced_feature,activation='selu')(x_global_max)
    x_dense_global_average_red= layers.Dense(reduced_feature,activation='selu')(x_global_average)
    #Feature_Reduction_Layer2
    features=output.shape[3]
    x_dense_global_max= layers.Dense(features,activation='selu')(x_dense_global_max_red)
    x_dense_global_average= layers.Dense(features,activation='selu')(x_dense_global_average_red)  
    #ChannelWiseAttention
    over_all_attention=layers.Activation('sigmoid')(x_dense_global_max+ x_dense_global_average)
    CBAM_Channel_output=layers.Multiply()([output,over_all_attention])
    return CBAM_Channel_output

def CBAM_Spatial(output,kernal:int=7):
    #Spatial_Attention
    #Maximum value across tensor
    x_global_max=tf.reduce_mean(output,axis=3,keepdims=True)
    #Average value across tensor
    x_global_average=tf.reduce_max(output,axis=3,keepdims=True)
    #Concatinate the feature maps
    concat=layers.Concatenate(axis=-1)([x_global_max,x_global_average])
    #Convolve with 7*7 filter
    conv=layers.Conv2D(1, (kernal, kernal), kernel_initializer='he_uniform', padding='same', use_bias=True)(concat)
    #BatchNorm
    conv=layers.BatchNormalization(axis=3)(conv)
    conv=layers.Activation('sigmoid')(conv)
    CBAM_Spatial_output=layers.Multiply()([output,conv])
    return CBAM_Spatial_output
from tensorflow.keras import layers

def residual_block(x, filters, kernel_size=5, strides=1):
    # Shortcut connection
    feature_map_channel=CBAM_Channel(x)
    feature_map_spatial=CBAM_Spatial(feature_map_channel)
    x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])

    shortcut = x
    
    # First convolutional layer
    
    x = layers.Conv2D(filters, kernel_size=kernel_size, padding='same', strides=strides)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('selu')(x)

    # Second convolutional layer
    feature_map_channel=CBAM_Channel(x)
    feature_map_spatial=CBAM_Spatial(feature_map_channel)
    x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])

    x = layers.Conv2D(filters, kernel_size=kernel_size, padding='same', strides=1)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('selu')(x)

    # Residual connection
    if strides != 1 or x.shape[-1] != shortcut.shape[-1]:
        #feature_map_channel=CBAM_Channel(x)
        #feature_map_spatial=CBAM_Spatial(feature_map_channel)
        #x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])

        shortcut = layers.Conv2D(filters, kernel_size=1, padding='same', strides=strides)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    feature_map_channel=CBAM_Channel(x)
    feature_map_spatial=CBAM_Spatial(feature_map_channel)
    x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])


    x = tf.keras.layers.Add()([x, shortcut])
    x = layers.Activation('selu')(x)

    return x


class Acgan:
    def __init__(self, eta, batch_size, epochs, weight_decay, latent_space,
                 image_shape, kernel_size):
        self.eta = eta
        self.batch_size = batch_size
        self.epochs = epochs
        self.weight_decay = weight_decay
        self.latent_space = latent_space
        self.image_shape = image_shape
        self.kernel_size = kernel_size
    def data(self, images, labels):
        ytrain = tf.keras.utils.to_categorical(labels)
        self.images = images
        self.labels = ytrain
    def samples(self, G, noize, labels):
        images = G.predict([noize, labels])
        ys = np.argmax(labels, axis = 1)
        plt.figure(figsize = (12, 4))
        for i in range(16):
            plt.subplot(2, 8, (i + 1))
            plt.imshow(images[i], cmap = 'gray')
            plt.title(ys[i])
        plt.show()
    def original_samples(self, X_train, y_train):
        '''
        images = G.predict([noize, labels])
        ys = np.argmax(labels, axis = 1)
        plt.figure(figsize = (12, 4))
        for i in range(16):
            plt.subplot(2, 8, (i + 1))
            plt.imshow(images[i], cmap = 'gray')
            plt.title(ys[i])
        plt.show()
        '''
        plt.figure(figsize = (12, 4))
        for i in range(16):
            plt.subplot(2, 8, (i + 1))
            plt.imshow(X_train[i], cmap = 'gray')
            plt.title(y_train[i])
        plt.show()
    
    '''def generator(self, inputs, labels):
        filters = [256, 128, 64, 32]
        padding = 'same'
        x = inputs
        y = labels
        x = layers.concatenate([x, y])
        x = layers.Dense(1024, )(x)
        x = layers.Dense(8*8*filters[0],
                         kernel_regularizer = tf.keras.regularizers.L2(0.001))(x)
        x = layers.Reshape((8, 8, filters[0]))(x)
        for filter in filters:
            if filter >= 64:
                strides = 2
            else:
                strides = 1
            x = LayerNormalization()(x)
            x = layers.Activation('relu')(x)
            x = Conv2DTranspose(filter, kernel_size = self.kernel_size, padding = padding,
                      strides = strides)(x)
        x = Conv2DTranspose(3, kernel_size = self.kernel_size, padding = padding)(x)
        x = layers.Activation('sigmoid')(x)
        self.generatorModel = models.Model(inputs = [inputs, labels],
                                           outputs = x,
                                           name = 'generator')'''
    def sample(self, z_mean, z_log_var):
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon
    
    def head1(self, x):
        x = residual_block(x, filters=16, strides=1)
        x = residual_block(x, filters=32, strides=1)
        
        x = residual_block(x, filters=64, strides=1)
        
        x = residual_block(x, filters=128, strides=1)
        
        x = residual_block(x, filters=256, strides=1)
        
        x = residual_block(x, filters=512, strides=1)
        
        # Additional three residual blocks with strides=1
        
        return x
    def head2(self, x):
        x = residual_block(x, filters=16, strides=1)
        x = residual_block(x, filters=32, strides=1)
        
        x = residual_block(x, filters=64, strides=1)
        
        x = residual_block(x, filters=128, strides=1)
        
        x = residual_block(x, filters=256, strides=1)
        
        x = residual_block(x, filters=512, strides=1)
        
        
        return x
    def head3(self, x):
        x = residual_block(x, filters=16, strides=1)
        x = residual_block(x, filters=32, strides=1)
        
        x = residual_block(x, filters=64, strides=1)
        
        x = residual_block(x, filters=128, strides=1)
        
        x = residual_block(x, filters=256, strides=1)
        
        x = residual_block(x, filters=512, strides=1)
        
        # Additional three residual blocks with strides=1
        
        return x
    def generator(self, inputs, labels):
        filters = [256, 128, 64, 32]
        padding = 'same'
        latent_dim = self.latent_space
        x = layers.concatenate([inputs, labels])
        x = layers.Dense(1024)(x)
        #changed
        #x = tf.keras.layers.Attention()([x, x])
        x = layers.Dense(8 * 8 * filters[0],
                         kernel_regularizer=tf.keras.regularizers.L2(0.001))(x)
        x = layers.Reshape((8, 8, filters[0]))(x)
        head_1 = x
        head_2 = x
        head_3 = x
        # Encoder
        #x = layers.concatenate([inputs, labels])
        #x = layers.Reshape((48, 48, 3))(x)  # Assuming input shape is (48, 48, 3)

        # Added Convolutional layers in the encoder
        '''feature_map_channel=CBAM_Channel(x)
        feature_map_spatial=CBAM_Spatial(feature_map_channel)
        x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])
        '''
        

        # Usage example for six residual blocks

        h1 = self.head1(head_1)
        print('h1:', h1.shape)
        h2 = self.head2(head_2)
        print('h2:', h2.shape)
        h3 = self.head3(head_3)
        print('h3:', h3.shape)
        mh = layers.concatenate([h1, h2, h3])
        print('mh:', mh.shape)
        
        feature_map_channel=CBAM_Channel(mh)
        feature_map_spatial=CBAM_Spatial(feature_map_channel)
        x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])
        
        x = layers.Conv2D(3, kernel_size=3, padding='same', strides=1)(x)
        x = layers.Activation('sigmoid')(x)
        #feature_map_channel=CBAM_Channel(x)
        #feature_map_spatial=CBAM_Spatial(feature_map_channel)
        #x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])
        

        x = layers.Flatten()(x)
        
        ## changed
        #x = tf.keras.layers.Attention()([x, x])
        
        x = layers.Dense(latent_dim * 2, activation = 'selu')(x)
        
        #x = tf.keras.layers.Attention()([x, x]) # Two times latent_dim for mean and log-variance
        
        x = layers.Reshape((2, latent_dim))(x)  # Split into mean and log-variance

        # Reparameterization trick
        z_mean, z_log_var = x[:, 0, :], x[:, 1, :]
        z = self.sample(z_mean, z_log_var)

        # Decoder
        x = layers.concatenate([z, labels])
        x = layers.Dense(32 * 32 * filters[0], kernel_regularizer=tf.keras.regularizers.L2(0.001))(x)
        
        #x = tf.keras.layers.Attention()([x, x])
        
        x = layers.Reshape((32, 32, filters[0]))(x)
        for filter in filters:
            if filter >= 128:
                strides = 2
            else:
                strides = 1
            x = layers.LayerNormalization()(x)
            x = layers.Activation('relu')(x)
            
            #feature_map_channel=CBAM_Channel(x)
            #feature_map_spatial=CBAM_Spatial(feature_map_channel)
            #x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])
            
            x = layers.Conv2DTranspose(filter, kernel_size=self.kernel_size, padding=padding,
                                       strides=strides)(x)
        x = layers.Conv2DTranspose(3, kernel_size=self.kernel_size, padding=padding)(x)
        
        #feature_map_channel=CBAM_Channel(x)
        #feature_map_spatial=CBAM_Spatial(feature_map_channel)
        #x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])
        
        x = layers.Activation('sigmoid')(x)

        self.generatorModel = models.Model(inputs=[inputs, labels], outputs=[x], name='generator')

    # Rest of the class definition and methods...

    def discriminator(self, inputs):
        x = inputs
        filters = [32, 64, 128, 256]
        padding = 'same'
        for filter in filters:
            if filter < 256:
                strides = 2
            else:
                strides = 1
            x = Conv2D(filter, kernel_size = self.kernel_size, padding = padding,
                      strides = strides,
                      kernel_regularizer = tf.keras.regularizers.L2(0.001))(x)
            x = LeakyReLU(alpha = 0.2)(x)
        x = layers.Flatten()(x)
        outputs = Dense(1, )(x)
        labelsOutput = Dense(256,
                             kernel_regularizer = tf.keras.regularizers.L2(0.001))(x)
        labelsOutput = Dropout(0.3)(labelsOutput)
        labelsOutput = Dense(2,)(labelsOutput)
        labelsOutput = layers.Activation('softmax')(labelsOutput)
        self.discriminatorModel = models.Model(inputs = inputs,
                                               outputs = [outputs, labelsOutput],
                                               name = 'discriminator')
    def build(self,):
        generatorInput = layers.Input(shape = (self.latent_space))
        discriminatorInput = layers.Input(shape = (self.image_shape))
        labelsInput = layers.Input(shape = (2, ))
        self.generator(generatorInput, labelsInput)
        self.discriminator(discriminatorInput)
        G = self.generatorModel
        D = self.discriminatorModel
        D.compile(loss = [combined_loss, 'binary_crossentropy'],
                 optimizer = tf.keras.optimizers.RMSprop(learning_rate = self.eta,
                                                        weight_decay = self.weight_decay))
        #D.summary()
        #G.summary()
        D.trainable = False
        GAN = models.Model(inputs = [generatorInput, labelsInput],
                           outputs = D(G([generatorInput, labelsInput])))
        GAN.compile(loss = [combined_loss, 'binary_crossentropy'],
                   optimizer = tf.keras.optimizers.RMSprop(learning_rate = self.eta*0.5,
                                                          weight_decay = self.weight_decay*0.5))
        GAN.summary()
        return G, D, GAN
    def trainAlgorithm(self, G, D, GAN):
        for epoch in range(self.epochs):
            indexs = np.random.randint(0, len(self.images), size = (self.batch_size, ))
            realImages = self.images[indexs]
            realLabels = self.labels[indexs]
            realTag = tf.ones(shape = (self.batch_size, ))
            noize = tf.random.uniform(shape = (self.batch_size,
                                              self.latent_space), minval = -1,
                                     maxval = 1)
            fakeLabels = tf.keras.utils.to_categorical(np.random.choice(range(2), size = (self.batch_size)),
                                                      num_classes = 2)
            fakeImages = tf.squeeze(G.predict([noize, fakeLabels], verbose = 0))
            #fakeImages = (G.predict([noize, fakeLabels], verbose = 0))
            #fakeImages, _, _ = G.predict([noize, fakeLabels], verbose=0)
            fakeTag = tf.zeros(shape = (self.batch_size, ))
            allImages = np.vstack([realImages, fakeImages])
            allLabels = np.vstack([realLabels, fakeLabels])
            allTags = np.hstack([realTag, fakeTag])
            _, dlossTag, dlossLabels = D.train_on_batch(allImages, [allTags, allLabels])
            noize = tf.random.uniform(shape = (self.batch_size,
                                              self.latent_space), minval = -1,
                                     maxval = 1)
            _, glossTag, glossLabels = GAN.train_on_batch([noize, fakeLabels], [realTag, fakeLabels])
            if epoch % 500 == 0:
                print('Epoch: {}'.format(epoch))
                print('discriminator loss: [tag: {}, labels: {}], generator loss: [tag: {}, labels: {}]'.format(dlossTag,
                                                                                                                dlossLabels,
                                                                                                                glossTag,
                                                                                                                glossLabels))
                self.samples(G, noize, fakeLabels)
                self.original_samples(images, labels)
            if epoch > 9997:
                print('Epoch: {}'.format(epoch))
                print('discriminator loss: [tag: {}, labels: {}], generator loss: [tag: {}, labels: {}]'.format(dlossTag,
                                                                                                                dlossLabels,
                                                                                                                glossTag,
                                                                                                                glossLabels))
                self.samples(G, noize, fakeLabels)
                self.original_samples(images, labels)

In [ ]:
import tensorflow as tf
def jensen_shannon_divergence(y_true, y_pred):
    # Ensure probabilities sum to 1
    y_true = tf.keras.backend.clip(y_true, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())
    y_pred = tf.keras.backend.clip(y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())

    # Calculate average probabilities
    avg_prob = (y_true + y_pred) / 2

    # Calculate Jensen-Shannon Divergence
    divergence = 0.5 * tf.keras.backend.sum(y_true * tf.keras.backend.log(y_true / avg_prob))
    divergence += 0.5 * tf.keras.backend.sum(y_pred * tf.keras.backend.log(y_pred / avg_prob))

    return divergence
import tensorflow as tf
def wasserstein_loss(y_true, y_pred):
    return tf.reduce_mean(y_true * y_pred)

def gradient_penalty_loss(y_true, y_pred, interpolated_samples):
    gradients = tf.gradients(y_pred, interpolated_samples)[0]
    gradients_sqr = tf.square(gradients)
    gradients_sqr_sum = tf.reduce_sum(gradients_sqr, axis=list(range(1, len(gradients_sqr.shape))))

    gradient_penalty = tf.sqrt(gradients_sqr_sum)
    return tf.reduce_mean(tf.square(1 - gradient_penalty))

def combined_loss(y_true, y_pred):
    # Adjust weights as needed
    mse_loss = tf.keras.losses.MeanSquaredError()(y_true, y_pred)
    kl_loss = tf.keras.losses.KLDivergence()(y_true, y_pred)
    #js_loss = jensen_shannon_divergence(y_true, y_pred)
    w_loss = wasserstein_loss(y_true, y_pred)

    # Generate interpolated samples for WGAN-GP
    #alpha = tf.random.uniform(shape=[tf.shape(y_true)[0], 1, 1, 1], minval=0.0, maxval=1.0)
    #interpolated_samples = alpha * y_true + (1 - alpha) * y_pred

    # Add Wasserstein loss and gradient penalty
    #w_loss = tf.reduce_mean(y_true) - tf.reduce_mean(y_pred)
    #gp_loss = gradient_penalty_loss(y_true, y_pred, interpolated_samples)

    # Adjust weights for Wasserstein loss and gradient penalty
    return 0.7 * mse_loss + 0.1 * kl_loss + 0.2 * w_loss

def CBAM_Channel(output,reduction:int=16):
    #Channel_Attention
    #GlobalMaxPooling of input tensor
    x_global_max=layers.GlobalMaxPooling2D()(output)
    #GlobalAveragePooling of input tensor
    x_global_average=layers.GlobalAveragePooling2D()(output)
    #Feature_Reduction_Layer1
    reduced_feature=output.shape[3]//reduction
    x_dense_global_max_red= layers.Dense(reduced_feature,activation='selu')(x_global_max)
    x_dense_global_average_red= layers.Dense(reduced_feature,activation='selu')(x_global_average)
    #Feature_Reduction_Layer2
    features=output.shape[3]
    x_dense_global_max= layers.Dense(features,activation='selu')(x_dense_global_max_red)
    x_dense_global_average= layers.Dense(features,activation='selu')(x_dense_global_average_red)  
    #ChannelWiseAttention
    over_all_attention=layers.Activation('sigmoid')(x_dense_global_max+ x_dense_global_average)
    CBAM_Channel_output=layers.Multiply()([output,over_all_attention])
    return CBAM_Channel_output

def CBAM_Spatial(output,kernal:int=7):
    #Spatial_Attention
    #Maximum value across tensor
    x_global_max=tf.reduce_mean(output,axis=3,keepdims=True)
    #Average value across tensor
    x_global_average=tf.reduce_max(output,axis=3,keepdims=True)
    #Concatinate the feature maps
    concat=layers.Concatenate(axis=-1)([x_global_max,x_global_average])
    #Convolve with 7*7 filter
    conv=layers.Conv2D(1, (kernal, kernal), kernel_initializer='he_uniform', padding='same', use_bias=True)(concat)
    #BatchNorm
    conv=layers.BatchNormalization(axis=3)(conv)
    conv=layers.Activation('sigmoid')(conv)
    CBAM_Spatial_output=layers.Multiply()([output,conv])
    return CBAM_Spatial_output
from tensorflow.keras import layers

def residual_block(x, filters, kernel_size=5, strides=1):
    # Shortcut connection
    feature_map_channel=CBAM_Channel(x)
    feature_map_spatial=CBAM_Spatial(feature_map_channel)
    x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])

    shortcut = x
    
    # First convolutional layer
    
    x = layers.Conv2D(filters, kernel_size=kernel_size, padding='same', strides=strides)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('selu')(x)

    # Second convolutional layer
    feature_map_channel=CBAM_Channel(x)
    feature_map_spatial=CBAM_Spatial(feature_map_channel)
    x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])

    x = layers.Conv2D(filters, kernel_size=kernel_size, padding='same', strides=1)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('selu')(x)

    # Residual connection
    if strides != 1 or x.shape[-1] != shortcut.shape[-1]:
        #feature_map_channel=CBAM_Channel(x)
        #feature_map_spatial=CBAM_Spatial(feature_map_channel)
        #x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])

        shortcut = layers.Conv2D(filters, kernel_size=1, padding='same', strides=strides)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    feature_map_channel=CBAM_Channel(x)
    feature_map_spatial=CBAM_Spatial(feature_map_channel)
    x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])


    x = tf.keras.layers.Add()([x, shortcut])
    x = layers.Activation('selu')(x)

    return x


class Acgan:
    def __init__(self, eta, batch_size, epochs, weight_decay, latent_space,
                 image_shape, kernel_size):
        self.eta = eta
        self.batch_size = batch_size
        self.epochs = epochs
        self.weight_decay = weight_decay
        self.latent_space = latent_space
        self.image_shape = image_shape
        self.kernel_size = kernel_size
    def data(self, images, labels):
        ytrain = tf.keras.utils.to_categorical(labels)
        self.images = images
        self.labels = ytrain
    def samples(self, G, noize, labels):
        images = G.predict([noize, labels])
        ys = np.argmax(labels, axis = 1)
        plt.figure(figsize = (12, 4))
        for i in range(16):
            plt.subplot(2, 8, (i + 1))
            plt.imshow(images[i], cmap = 'gray')
            plt.title(ys[i])
        plt.show()
    def original_samples(self, X_train, y_train):
        '''
        images = G.predict([noize, labels])
        ys = np.argmax(labels, axis = 1)
        plt.figure(figsize = (12, 4))
        for i in range(16):
            plt.subplot(2, 8, (i + 1))
            plt.imshow(images[i], cmap = 'gray')
            plt.title(ys[i])
        plt.show()
        '''
        plt.figure(figsize = (12, 4))
        for i in range(16):
            plt.subplot(2, 8, (i + 1))
            plt.imshow(X_train[i], cmap = 'gray')
            plt.title(y_train[i])
        plt.show()
    
    '''def generator(self, inputs, labels):
        filters = [256, 128, 64, 32]
        padding = 'same'
        x = inputs
        y = labels
        x = layers.concatenate([x, y])
        x = layers.Dense(1024, )(x)
        x = layers.Dense(8*8*filters[0],
                         kernel_regularizer = tf.keras.regularizers.L2(0.001))(x)
        x = layers.Reshape((8, 8, filters[0]))(x)
        for filter in filters:
            if filter >= 64:
                strides = 2
            else:
                strides = 1
            x = LayerNormalization()(x)
            x = layers.Activation('relu')(x)
            x = Conv2DTranspose(filter, kernel_size = self.kernel_size, padding = padding,
                      strides = strides)(x)
        x = Conv2DTranspose(3, kernel_size = self.kernel_size, padding = padding)(x)
        x = layers.Activation('sigmoid')(x)
        self.generatorModel = models.Model(inputs = [inputs, labels],
                                           outputs = x,
                                           name = 'generator')'''
    def sample(self, z_mean, z_log_var):
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon
    
    def head1(self, x):
        x = residual_block(x, filters=16, strides=1)
        x = residual_block(x, filters=16, strides=1)
        x = residual_block(x, filters=32, strides=1)
        x = residual_block(x, filters=32, strides=1)
        x = residual_block(x, filters=64, strides=1)
        x = residual_block(x, filters=64, strides=1)
        
        x = residual_block(x, filters=128, strides=1)
        x = residual_block(x, filters=128, strides=1)
        x = residual_block(x, filters=256, strides=1)
        x = residual_block(x, filters=256, strides=1)
        x = residual_block(x, filters=512, strides=1)
        x = residual_block(x, filters=512, strides=1)
        
        return x
    def head2(self, x):
        x = residual_block(x, filters=16, strides=1)
        x = residual_block(x, filters=16, strides=1)
        x = residual_block(x, filters=32, strides=1)
        x = residual_block(x, filters=32, strides=1)
        x = residual_block(x, filters=64, strides=1)
        x = residual_block(x, filters=64, strides=1)
        
        x = residual_block(x, filters=128, strides=1)
        x = residual_block(x, filters=128, strides=1)
        x = residual_block(x, filters=256, strides=1)
        x = residual_block(x, filters=256, strides=1)
        x = residual_block(x, filters=512, strides=1)
        x = residual_block(x, filters=512, strides=1)
        
        
        return x
    def head3(self, x):
        x = residual_block(x, filters=16, strides=1)
        x = residual_block(x, filters=16, strides=1)
        x = residual_block(x, filters=32, strides=1)
        x = residual_block(x, filters=32, strides=1)
        x = residual_block(x, filters=64, strides=1)
        x = residual_block(x, filters=64, strides=1)
        
        x = residual_block(x, filters=128, strides=1)
        x = residual_block(x, filters=128, strides=1)
        x = residual_block(x, filters=256, strides=1)
        x = residual_block(x, filters=256, strides=1)
        x = residual_block(x, filters=512, strides=1)
        x = residual_block(x, filters=512, strides=1)
        
        # Additional three residual blocks with strides=1
        
        return x
    def generator(self, inputs, labels):
        filters = [256, 128, 64, 32]
        padding = 'same'
        latent_dim = self.latent_space
        x = layers.concatenate([inputs, labels])
        x = layers.Dense(1024)(x)
        x=tf.keras.layers.Attention()([x,x])
        x = layers.Dense(8 * 8 * filters[0],
                         kernel_regularizer=tf.keras.regularizers.L2(0.001))(x)
        x=tf.keras.layers.Attention()([x,x])
        x = layers.Reshape((8, 8, filters[0]))(x)
        head_1 = x
        head_2 = x
        head_3 = x
        # Encoder
        #x = layers.concatenate([inputs, labels])
        #x = layers.Reshape((48, 48, 3))(x)  # Assuming input shape is (48, 48, 3)

        # Added Convolutional layers in the encoder
        '''feature_map_channel=CBAM_Channel(x)
        feature_map_spatial=CBAM_Spatial(feature_map_channel)
        x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])
        '''
        

        # Usage example for six residual blocks

        h1 = self.head1(head_1)
        print('h1:', h1.shape)
        h2 = self.head2(head_2)
        print('h2:', h2.shape)
        h3 = self.head3(head_3)
        print('h3:', h3.shape)
        mh = layers.concatenate([h1, h2, h3])
        print('mh:', mh.shape)
        
        feature_map_channel=CBAM_Channel(mh)
        feature_map_spatial=CBAM_Spatial(feature_map_channel)
        x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])
        
        x = layers.Conv2D(3, kernel_size=3, padding='same', strides=2)(x)
        x = layers.Activation('sigmoid')(x)

        x = layers.Flatten()(x)
        x = tf.keras.layers.Attention()([x,x])
        
        x = layers.Dense(latent_dim * 2, activation = 'selu')(x)  # Two times latent_dim for mean and log-variance
        x=tf.keras.layers.Attention()([x,x])
        
        x = layers.Reshape((2, latent_dim))(x)  # Split into mean and log-variance

        # Reparameterization trick
        z_mean, z_log_var = x[:, 0, :], x[:, 1, :]
        z = self.sample(z_mean, z_log_var)

        # Decoder
        x = layers.concatenate([z, labels])
        x = layers.Dense(32 * 32 * filters[0], kernel_regularizer=tf.keras.regularizers.L2(0.001))(x)
        x = tf.keras.layers.Attention()([x,x])
        
        x = layers.Reshape((32, 32, filters[0]))(x)
        x1 = x
        x2 = x
        x3 = x
        
        for filter in filters:
            if filter >= 128:
                strides = 2
            else:
                strides = 1
            x1 = layers.LayerNormalization()(x1)
            x1 = layers.Activation('selu')(x1)
            
            feature_map_channel=CBAM_Channel(x1)
            feature_map_spatial=CBAM_Spatial(feature_map_channel)
            x1 = tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])

            x1 = layers.Conv2DTranspose(filter, kernel_size=self.kernel_size, padding=padding,
                                       strides=strides)(x1)
        #x1 = layers.Conv2DTranspose(3, kernel_size=self.kernel_size, padding=padding)(x1)
        #feature_map_channel=CBAM_Channel(x1)
        #feature_map_spatial=CBAM_Spatial(feature_map_channel)
        #x1 = tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])
        
        for filter in filters:
            if filter >= 128:
                strides = 2
            else:
                strides = 1
            x2 = layers.LayerNormalization()(x2)
            x2 = layers.Activation('selu')(x2)
            
            feature_map_channel=CBAM_Channel(x2)
            feature_map_spatial=CBAM_Spatial(feature_map_channel)
            x2 = tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])

            x2 = layers.Conv2DTranspose(filter, kernel_size=self.kernel_size, padding=padding,
                                       strides=strides)(x2)
        #x2 = layers.Conv2DTranspose(3, kernel_size=self.kernel_size, padding=padding)(x2)
        
        '''
        for filter in filters:
            if filter >= 128:
                strides = 2
            else:
                strides = 1
            x3 = layers.LayerNormalization()(x3)
            x3 = layers.Activation('selu')(x3)
            
            feature_map_channel=CBAM_Channel(x3)
            feature_map_spatial=CBAM_Spatial(feature_map_channel)
            x3 = tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])

            x3 = layers.Conv2DTranspose(filter, kernel_size=self.kernel_size, padding=padding,
                                       strides=strides)(x3)
        x3 = layers.Conv2DTranspose(3, kernel_size=self.kernel_size, padding=padding)(x3)
        feature_map_channel=CBAM_Channel(x3)
        feature_map_spatial=CBAM_Spatial(feature_map_channel)
        x3 = tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])
        '''

        x = layers.concatenate([x1, x2])
        feature_map_channel=CBAM_Channel(x)
        feature_map_spatial=CBAM_Spatial(feature_map_channel)
        x=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])
        
        
        x = layers.Conv2DTranspose(3, kernel_size=self.kernel_size, padding=padding)(x)
        x = layers.Activation('sigmoid')(x)
        
        
        
        
        self.generatorModel = models.Model(inputs=[inputs, labels], outputs=[x], name='generator')

    # Rest of the class definition and methods...

    def discriminator(self, inputs):
        x1 = inputs
        x2 = inputs
        filters = [32, 64, 128, 256]
        padding = 'same'
        for filter in filters:
            if filter < 256:
                strides = 2
            else:
                strides = 1
                
            feature_map_channel=CBAM_Channel(x1)
            feature_map_spatial=CBAM_Spatial(feature_map_channel)
            x1=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])

            x1 = Conv2D(filter, kernel_size = self.kernel_size, padding = padding,
                      strides = strides, kernel_regularizer = tf.keras.regularizers.L2(0.001))(x1)
            x1 = LeakyReLU(alpha = 0.2)(x1)
        
        for filter in filters:
            if filter < 256:
                strides = 2
            else:
                strides = 1
            
            feature_map_channel=CBAM_Channel(x2)
            feature_map_spatial=CBAM_Spatial(feature_map_channel)
            x2=tf.keras.layers.Add()([feature_map_channel,feature_map_spatial])

            x2 = Conv2D(filter, kernel_size = self.kernel_size, padding = padding,
                      strides = strides, kernel_regularizer = tf.keras.regularizers.L2(0.001))(x2)
            x2 = LeakyReLU(alpha = 0.2)(x2)
            
        x = layers.concatenate([x1, x2]) ### mh = layers.concatenate([h1, h2, h3])
        x = layers.Flatten()(x)
        outputs = Dense(1, )(x)
        labelsOutput = Dense(256,
                             kernel_regularizer = tf.keras.regularizers.L2(0.001))(x)
        labelsOutput = Dropout(0.3)(labelsOutput)
        labelsOutput = Dense(2,)(labelsOutput)
        labelsOutput = layers.Activation('softmax')(labelsOutput)
        self.discriminatorModel = models.Model(inputs = inputs,
                                               outputs = [outputs, labelsOutput],
                                               name = 'discriminator')
    def build(self,):
        generatorInput = layers.Input(shape = (self.latent_space))
        discriminatorInput = layers.Input(shape = (self.image_shape))
        labelsInput = layers.Input(shape = (2, ))
        self.generator(generatorInput, labelsInput)
        self.discriminator(discriminatorInput)
        G = self.generatorModel
        D = self.discriminatorModel
        D.compile(loss = [combined_loss, 'binary_crossentropy'],
                 optimizer = tf.keras.optimizers.RMSprop(learning_rate = self.eta,
                                                        weight_decay = self.weight_decay))
        #D.summary()
        #G.summary()
        D.trainable = False
        GAN = models.Model(inputs = [generatorInput, labelsInput],
                           outputs = D(G([generatorInput, labelsInput])))
        GAN.compile(loss = [combined_loss, 'binary_crossentropy'],
                   optimizer = tf.keras.optimizers.RMSprop(learning_rate = self.eta*0.5,
                                                          weight_decay = self.weight_decay*0.5))
        GAN.summary()
        return G, D, GAN
    def trainAlgorithm(self, G, D, GAN):
        for epoch in range(self.epochs):
            indexs = np.random.randint(0, len(self.images), size = (self.batch_size, ))
            realImages = self.images[indexs]
            realLabels = self.labels[indexs]
            realTag = tf.ones(shape = (self.batch_size, ))
            noize = tf.random.uniform(shape = (self.batch_size,
                                              self.latent_space), minval = -1,
                                     maxval = 1)
            fakeLabels = tf.keras.utils.to_categorical(np.random.choice(range(2), size = (self.batch_size)),
                                                      num_classes = 2)
            fakeImages = tf.squeeze(G.predict([noize, fakeLabels], verbose = 0))
            #fakeImages = (G.predict([noize, fakeLabels], verbose = 0))
            #fakeImages, _, _ = G.predict([noize, fakeLabels], verbose=0)
            fakeTag = tf.zeros(shape = (self.batch_size, ))
            allImages = np.vstack([realImages, fakeImages])
            allLabels = np.vstack([realLabels, fakeLabels])
            allTags = np.hstack([realTag, fakeTag])
            _, dlossTag, dlossLabels = D.train_on_batch(allImages, [allTags, allLabels])
            noize = tf.random.uniform(shape = (self.batch_size,
                                              self.latent_space), minval = -1,
                                     maxval = 1)
            _, glossTag, glossLabels = GAN.train_on_batch([noize, fakeLabels], [realTag, fakeLabels])
            if epoch % 500 == 0:
                print('Epoch: {}'.format(epoch))
                print('discriminator loss: [tag: {}, labels: {}], generator loss: [tag: {}, labels: {}]'.format(dlossTag,
                                                                                                                dlossLabels,
                                                                                                                glossTag,
                                                                                                                glossLabels))
                self.samples(G, noize, fakeLabels)
                self.original_samples(images, labels)
            if epoch > 9997:
                print('Epoch: {}'.format(epoch))
                print('discriminator loss: [tag: {}, labels: {}], generator loss: [tag: {}, labels: {}]'.format(dlossTag,
                                                                                                                dlossLabels,
                                                                                                                glossTag,
                                                                                                                glossLabels))
                self.samples(G, noize, fakeLabels)
                self.original_samples(images, labels)

- In order to avoid falling into the problem of mode collapse, a smaller number of samples was used to be passed each time to the generative adversarial network.
- It helped reduce the number of samples that are passed to the obstetric adversarial network, until the generator collects more accurately the areas that affect the presence of pneumonia in the patient, as well as the healthy condition.

- Because I didn't have much computational resources, not many Epochs were used.

- Since we are dealing with a pathological condition, the use of kernel_size in a larger size helps to study the relationship between the core regions and the surrounding areas, and the possibility of the existence of gradients that express the pathological condition or the healthy condition. (That is, it helps to determine whether an area, according to its location, can be suitable as a criterion for the presence of pneumonia, or not).
- Using LayerNormalization instead of BatchNormalization was very useful in diversifying the images that the generator generates and not falling into mode collapse (since LayerNormalization does the normalization at the level of the filters included in the layer).

In [ ]:
acgan = Acgan(eta = 0.00001, batch_size = 32, epochs = 10000, weight_decay = 6e-5,
              latent_space = 128, image_shape = (128, 128, 3), kernel_size = 5)

In [ ]:
acgan.data(images, labels)

In [ ]:
G, D, GAN = acgan.build()

In [ ]:
acgan.trainAlgorithm(G, D, GAN)

In [ ]:
acgan.trainAlgorithm(G, D, GAN)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_resnet18(input_shape=(128, 128, 3), num_classes=3):
    input_tensor = tf.keras.Input(shape=input_shape)
    
    # Initial Convolution
    x = layers.Conv2D(64, (7, 7), strides=(2, 2), padding='same', activation='relu')(input_tensor)
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same')(x)
    
    # Residual Blocks
    x = residual_block(x, filters=[64, 64], kernel_size=3, strides=1, block_name='resblock1')
    x = residual_block(x, filters=[128, 128], kernel_size=3, strides=2, block_name='resblock2')
    x = residual_block(x, filters=[256, 256], kernel_size=3, strides=2, block_name='resblock3')
    x = residual_block(x, filters=[512, 512], kernel_size=3, strides=2, block_name='resblock4')
    
    # Global Average Pooling
    x = layers.GlobalAveragePooling2D()(x)
    
    # Fully Connected Layer
    x = layers.Dense(128, name='dense1', activation='relu')(x)
    x = layers.Dense(2, activation='sigmoid')(x)

    model = models.Model(inputs=input_tensor, outputs=x, name='resnet18')
    
    return model

def residual_block(x, filters, kernel_size=3, strides=1, block_name='resblock'):
    shortcut = x
    
    # First convolution
    x = layers.Conv2D(filters[0], (kernel_size, kernel_size), strides=(strides, strides), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Second convolution
    x = layers.Conv2D(filters[1], (kernel_size, kernel_size), strides=(1, 1), padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Shortcut connection
    if strides != 1 or shortcut.shape[-1] != filters[1]:
        shortcut = layers.Conv2D(filters[1], (1, 1), strides=(strides, strides), padding='valid')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # Add shortcut to the main path
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    return x

# Example usage
resnet18_model = build_resnet18()
resnet18_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# Example: Assuming you have X_train and y_train
#import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]
one_hot_labels = tf.one_hot(labels, depth=2)
resnet18_model.fit(images, one_hot_labels, epochs=100, callbacks = callbacks, 
        
          validation_split=0.2)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNet, VGG16
from tensorflow.keras.layers import Layer, Conv2D, DepthwiseConv2D

# Assume you have defined CombinedAttentionNoiseLayer class

# Load pre-trained MobileNet model (excluding top layers)
input_shape = (128, 128, 3)
mobilenet_base = MobileNet(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the layers of MobileNet
for layer in mobilenet_base.layers:
    layer.trainable = False

# Specify the indices of layers where you want to add combined attention noise
#
# Create custom attention noise layers
input_data = Input(shape=input_shape, name='input_data')
#attention_noise_output = CombinedAttentionNoiseLayer(spatial_noise_factor=0.1, 
 #                                                    channel_noise_factor=0.1)(input_data)

# Apply combined attention noise before specified layers
x = input_data
for i, layer in enumerate(mobilenet_base.layers):
    #if i in attention_indices:
        #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, channel_noise_factor=1.0)(x)
    x = layer(x)

# Add additional layers for classification
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='selu')(x)
output_layer = Dense(3, activation='softmax')(x)

# Create the final model
model = Model(inputs=input_data, outputs=output_layer)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()
#model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()

# Continue with training the model...
# Continue with training the model...

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]
one_hot_labels = tf.one_hot(labels, depth=3)
model.fit(images, one_hot_labels, epochs=100, callbacks = callbacks, 
        
          validation_split=0.2)

# Continue with training the model...


In [ ]:
import pandas as pd
reshaped_tensor = tf.reduce_mean(images_test, axis=(2, 3))
print(reshaped_tensor.shape)

#one_hot_encoded = pd.get_dummies(labels_test1)
one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
print(one_hot_encoded.shape)
model.evaluate(images_test, one_hot_encoded)
print(reshaped_tensor.shape, one_hot_encoded.shape)

#denoised_features_fgsm = G.predict([reshaped_tensor, one_hot_encoded])
#print(denoised_features_fgsm.shape)

denoised_images_pgd = G.predict([reshaped_tensor, one_hot_encoded]) #G1.predict([reshaped_tensor, one_hot_encoded])
print(denoised_images_pgd.shape)
model.evaluate(denoised_images_pgd, one_hot_encoded)

In [ ]:
import pandas as pd
reshaped_tensor = tf.reduce_mean(images_test, axis=(2, 3))
print(reshaped_tensor.shape)

#one_hot_encoded = pd.get_dummies(labels_test1)
one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
print(one_hot_encoded.shape)
resnet18_model.evaluate(images_test, one_hot_encoded)
print(reshaped_tensor.shape, one_hot_encoded.shape)

#denoised_features_fgsm = G.predict([reshaped_tensor, one_hot_encoded])
#print(denoised_features_fgsm.shape)

denoised_images_pgd = G.predict([reshaped_tensor, one_hot_encoded]) #G1.predict([reshaped_tensor, one_hot_encoded])
print(denoised_images_pgd.shape)
resnet18_model.evaluate(denoised_images_pgd, one_hot_encoded)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNet, VGG16, ResNet50, MobileNetV2
from tensorflow.keras.layers import Layer, Conv2D, DepthwiseConv2D

# Assume you have defined CombinedAttentionNoiseLayer class

# Load pre-trained MobileNet model (excluding top layers)
input_shape = (128, 128, 3)
mobilenet_base = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the layers of MobileNet
for layer in mobilenet_base.layers:
    layer.trainable = False

# Specify the indices of layers where you want to add combined attention noise
#
# Create custom attention noise layers
input_data = Input(shape=input_shape, name='input_data')
#attention_noise_output = CombinedAttentionNoiseLayer(spatial_noise_factor=0.1, 
 #                                                    channel_noise_factor=0.1)(input_data)

# Apply combined attention noise before specified layers
x = input_data
#for i, layer in enumerate(mobilenet_base.layers):
    #if i in attention_indices:
        #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, channel_noise_factor=1.0)(x)
    #x = layer(x)

# Add additional layers for classification
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='selu')(x)
output_layer = Dense(2, activation='sigmoid')(x)

# Create the final model
model = Model(inputs=input_data, outputs=output_layer)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=300)
callbacks = [checkpoint_callback, early_stopping]
one_hot_labels = tf.one_hot(labels, depth=2)
model.fit(images, one_hot_labels, epochs=300, callbacks = callbacks, 
          validation_split=0.2)

# Continue with training the model...


# Print the model summary
#model.summary()
#model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()

# Continue with training the model...
# Continue with training the model...

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=300)
callbacks = [checkpoint_callback, early_stopping]
one_hot_labels = tf.one_hot(labels, depth=2)
model.fit(images, one_hot_labels, epochs=300, callbacks = callbacks, batch_size=128,
          validation_split=0.2)

# Continue with training the model...


In [ ]:
import pandas as pd
reshaped_tensor = tf.reduce_mean(images_test, axis=(2, 3))
print(reshaped_tensor.shape)

#one_hot_encoded = pd.get_dummies(labels_test1)
one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
print(one_hot_encoded.shape)
model.evaluate(images_test, one_hot_encoded)
print(reshaped_tensor.shape, one_hot_encoded.shape)

#denoised_features_fgsm = G.predict([reshaped_tensor, one_hot_encoded])
#print(denoised_features_fgsm.shape)

denoised_images_pgd = G.predict([reshaped_tensor, one_hot_encoded]) #G1.predict([reshaped_tensor, one_hot_encoded])
print(denoised_images_pgd.shape)
model.evaluate(denoised_images_pgd, one_hot_encoded)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Load pre-trained ResNet50 model without the top layer1
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# Freeze the pre-trained layers
for layer in base_model.layers:
    layer.trainable = False

# Create a new model with a custom top layer for binary classification
model1 = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='selu'),
    Dense(2, activation='sigmoid')  # Binary classification, so use sigmoid activation
])

# Compile the model
model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]
one_hot_labels = tf.one_hot(labels, depth=2)
model1.fit(images, one_hot_labels, epochs=100, callbacks = callbacks, 
          validation_split=0.2)

# Continue with training the model...


# Display the model summary
#


In [ ]:
import pandas as pd
reshaped_tensor = tf.reduce_mean(images_test, axis=(2, 3))
print(reshaped_tensor.shape)

#one_hot_encoded = pd.get_dummies(labels_test1)
one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
print(one_hot_encoded.shape)
model1.evaluate(images_test, one_hot_encoded)
print(reshaped_tensor.shape, one_hot_encoded.shape)

#denoised_features_fgsm = G.predict([reshaped_tensor, one_hot_encoded])
#print(denoised_features_fgsm.shape)

denoised_images_pgd = G.predict([reshaped_tensor, one_hot_encoded]) #G1.predict([reshaped_tensor, one_hot_encoded])
print(denoised_images_pgd.shape)
model1.evaluate(denoised_images_pgd, one_hot_encoded)

In [ ]:
import pandas as pd
reshaped_tensor = tf.reduce_mean(images_test, axis=(2, 3))
print(reshaped_tensor.shape)

#one_hot_encoded = pd.get_dummies(labels_test1)
one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
print(one_hot_encoded.shape)
resnet18_model.evaluate(images_test, one_hot_encoded)
print(reshaped_tensor.shape, one_hot_encoded.shape)

#denoised_features_fgsm = G.predict([reshaped_tensor, one_hot_encoded])
#print(denoised_features_fgsm.shape)

denoised_images_pgd = G.predict([reshaped_tensor, one_hot_encoded]) #G1.predict([reshaped_tensor, one_hot_encoded])
print(denoised_images_pgd.shape)
resnet18_model.evaluate(denoised_images_pgd, one_hot_encoded)

In [ ]:
G1 = G

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Access the second last dense layer by name
second_last_dense_layer = model.get_layer('dense_155')

# Alternatively, you can access the second last layer by index
# Index -2 corresponds to the second last layer
second_last_dense_layer = model.get_layer('dense_155')

# Create a sub-model to obtain latent code
latent_code_model = models.Model(inputs=model.input, outputs=second_last_dense_layer.output)

# Assuming you have an input image 'input_image'
latent_code = latent_code_model.predict(images_test, batch_size=8)
print("Latent Code Shape:", latent_code.shape)

import pandas as pd
one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
print(one_hot_encoded.shape)

denoised_images_pgd = G.predict([latent_code, one_hot_encoded]) #G1.predict([reshaped_tensor, one_hot_encoded])
print(denoised_images_pgd.shape)

In [ ]:
import numpy as np

epochs = 100
batch_size = 64

def get_real_batch(x, batch_size):
    indices = np.random.choice(len(x), batch_size, replace=False)
    real_images = x[indices]
    return real_images

# Assuming you have your original GAN training loop
for epoch in range(epochs):
    print('epoch:', epoch)
    for i in range(len(images) // batch_size):
        print('i:', i)
        # Get a batch of real examples
        real_examples = get_real_batch(images, batch_size)
        second_last_dense_layer = resnet18_model.get_layer('dense1')

        # Alternatively, you can access the second last layer by index
        # Index -2 corresponds to the second last layer
        #second_last_dense_layer = model.get_layer('dense_156')
    
        # Create a sub-model to obtain latent code
        latent_code_model = models.Model(inputs=resnet18_model.input, 
                                         outputs=second_last_dense_layer.output)
        
        # Assuming you have an input image 'input_image'
        latent_code = latent_code_model.predict(images)
        print("Latent Code Shape:", latent_code.shape)
        
        from tensorflow.keras.utils import to_categorical
        one_hot_labels = to_categorical(labels, num_classes=2)
        print("one_hot_labels  Shape:", one_hot_labels.shape)
        
        
        # Generate denoised examples using the current latent code
        denoised_examples = G.predict([latent_code, one_hot_labels])
        #denoised_images_pgd = G.predict([latent_code, one_hot_encoded]) #G1.predict([reshaped_tensor, one_hot_encoded])
        print('denoised_examples:',denoised_examples.shape)
        resnet18_model.evaluate(denoised_examples, one_hot_labels)

        # Get a batch of denoised examples
        denoised_examples_batch = get_real_batch(denoised_examples, batch_size)

        # Train the discriminator on real examples
        '''
        realLabels = self.labels[indexs]
            realTag = tf.ones(shape = (self.batch_size, ))
        fakeLabels = tf.keras.utils.to_categorical(np.random.choice(range(3), size = (self.batch_size)),
                                                      num_classes = 3)
            fakeImages = tf.squeeze(G.predict([noize, fakeLabels], verbose = 0))
            #fakeImages = (G.predict([noize, fakeLabels], verbose = 0))
            #fakeImages, _, _ = G.predict([noize, fakeLabels], verbose=0)
            fakeTag = tf.zeros(shape = (self.batch_size, ))
            allImages = np.vstack([realImages, fakeImages])
            allLabels = np.vstack([realLabels, fakeLabels])
            allTags = np.hstack([realTag, fakeTag])
            _, dlossTag, dlossLabels = D.train_on_batch(allImages, [allTags, allLabels])
        '''
        #fakeLabels = one_hot_labels
        fakeLabels = get_real_batch(one_hot_labels, batch_size)
        #fakeTag = tf.zeros(shape = (batch_size, ))
        fakeTag = tf.concat([tf.zeros((batch_size, 1))
                                               , tf.zeros((batch_size, 1))
                                                      ], axis=1)
        realLabels = get_real_batch(one_hot_labels, batch_size)
        
        #realTag = tf.ones(shape = (batch_size, ))
        realTag = tf.concat([tf.ones((batch_size, 1))
                                               , tf.ones((batch_size, 1))
                                                      ], axis=1)
        
        d_loss_real = D.train_on_batch(real_examples,[realTag, realLabels])

        # Train the discriminator on denoised examples
        d_loss_fake = D.train_on_batch(denoised_examples_batch, [fakeTag, fakeLabels])

        # Calculate the total discriminator loss
        d_loss_real_value = d_loss_real[0]
        d_loss_fake_value = d_loss_fake[0]

        # Calculate the total discriminator loss
        d_loss = 0.5 * (d_loss_real_value + d_loss_fake_value)

        # Train the generator to fool the discriminator
        latent_code_examples = get_real_batch(latent_code, batch_size)
        one_hot_labels_examples = get_real_batch(one_hot_labels, batch_size)
        #_, glossTag, glossLabels = GAN.train_on_batch([noize, fakeLabels], [realTag, fakeLabels])
        g_loss = GAN.train_on_batch([latent_code_examples, one_hot_labels_examples], 
                                    [realTag, one_hot_labels_examples])

    print(f"Epoch {epoch + 1}, D Loss: {d_loss}, G Loss: {g_loss}")

# After the training loop, you can use the updated generator for denoising
denoised_image = G.predict([latent_code, one_hot_labels])
print("Denoised Image Shape:", denoised_image.shape)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Access the second last dense layer by name
second_last_dense_layer = resnet18_model.get_layer('dense1')

# Alternatively, you can access the second last layer by index
# Index -2 corresponds to the second last layer
second_last_dense_layer = resnet18_model.get_layer('dense1')

# Create a sub-model to obtain latent code
latent_code_model = models.Model(inputs=resnet18_model.input, outputs=second_last_dense_layer.output)

# Assuming you have an input image 'input_image'
latent_code = latent_code_model.predict(images_test)
print("Latent Code Shape:", latent_code.shape)

import pandas as pd
one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
print(one_hot_encoded.shape)

denoised_images_pgd = G.predict([latent_code, one_hot_encoded]) #G1.predict([reshaped_tensor, one_hot_encoded])
print(denoised_images_pgd.shape)
resnet18_model.evaluate(denoised_images_pgd, one_hot_encoded)

In [ ]:
from tensorflow.keras.utils import to_categorical
one_hot_labels = to_categorical(labels_test, num_classes=3)

In [ ]:
epochs = 100
batch_size = 128

def get_real_batch(batch_size, x):
    indices = np.random.choice(len(x), batch_size, replace=False)
    real_images = x[indices]
    return real_images

# Assuming you have your original GAN training loop
epochs = 100
batch_size = 128

for epoch in range(epochs):
    for _ in range(len(images) // batch_size):
        # Get a batch of real examples
        real_examples = get_real_batch(batch_size, images)  # Implement your function to get a batch of real examples
        import tensorflow as tf
        from tensorflow.keras import layers, models

        # Access the second last dense layer by name
        second_last_dense_layer = resnet18_model.get_layer('dense1')

        # Alternatively, you can access the second last layer by index
        # Index -2 corresponds to the second last layer
        second_last_dense_layer = resnet18_model.get_layer('dense1')

        # Create a sub-model to obtain latent code
        latent_code_model = models.Model(inputs=resnet18_model.input, outputs=second_last_dense_layer.output)

        # Assuming you have an input image 'input_image'
        latent_code = latent_code_model.predict(images, batch_size=8)
        print("Latent Code Shape:", latent_code.shape)
        from tensorflow.keras.utils import to_categorical
        one_hot_labels = to_categorical(labels, num_classes=3)
        # Generate denoised examples using the current latent code
        denoised_examples = G.predict([latent_code, one_hot_labels])
        #resnet18_model.evaluate(denoised_examples, one_hot_labels)
        d_examples = get_real_batch(batch_size, denoised_examples)

        # Train the discriminator
        '''
        d_loss_real = D.train_on_batch(real_examples, tf.ones((batch_size, 1)))
        d_loss_fake = D.train_on_batch(denoised_examples, tf.zeros((batch_size, 1)))
        d_loss = 0.5 * (d_loss_real + d_loss_fake)
        '''
        # Train the discriminator on real examples
        d_loss_real = D.train_on_batch(real_examples, tf.concat([tf.ones((batch_size, 1)), 
                                                                 tf.zeros((batch_size, 1))], axis=1))

        # Train the discriminator on denoised examples
        d_loss_fake = D.train_on_batch(d_examples, tf.concat([tf.zeros((batch_size, 1)), 
                                                              tf.ones((batch_size, 1))], axis=1))

        # Calculate the total discriminator loss
        d_loss_real_value = d_loss_real[0]
        d_loss_fake_value = d_loss_fake[0]

        # Calculate the total discriminator loss
        d_loss = 0.5 * (d_loss_real_value + d_loss_fake_value)
            

        # Train the generator to fool the discriminator
        g_loss = GAN.train_on_batch([latent_code, one_hot_labels], tf.ones((batch_size, 1)))

    print(f"Epoch {epoch + 1}, D Loss: {d_loss}, G Loss: {g_loss}")

# After the training loop, you can use the updated generator for denoising
denoised_image = G.predict([latent_code, one_hot_labels])
print("Denoised Image Shape:", denoised_image.shape)

In [ ]:
import numpy as np

# Assuming X_train is a numpy array containing your training images

# Get the number of images in X_train
num_images = images.shape[0]

# Generate 1000 random indices between 0 and num_images - 1
random_indices = np.random.choice(num_images, 1000, replace=False)

# Select the images corresponding to these random indices
random_samples = images[random_indices]
labels1 = labels[random_indices]
random_samples.shape, labels1.shape
# Now random_samples contains the randomly selected 1000 samples from X_train


In [ ]:
from tensorflow.keras.utils import to_categorical
one_hot_labels = to_categorical(labels1, num_classes=2)

resnet18_model.evaluate(random_samples, one_hot_labels)

second_last_dense_layer = resnet18_model.get_layer('dense1')

latent_code_model = models.Model(inputs=resnet18_model.input, outputs=second_last_dense_layer.output)

# Assuming you have an input image 'input_image'
latent_code = latent_code_model.predict(images_test)
print("Latent Code Shape:", latent_code.shape)

one_hot_labels_test = to_categorical(labels_test, num_classes=2)
denoised_images_pgd = G.predict([latent_code, one_hot_labels_test]) #G1.predict([reshaped_tensor, one_hot_encoded])
print(denoised_images_pgd.shape)
resnet18_model.evaluate(denoised_images_pgd, one_hot_labels_test)

In [ ]:
import numpy as np

epochs = 1
batch_size = 16

def get_real_batch(x, batch_size):
    indices = np.random.choice(len(x), batch_size, replace=False)
    real_images = x[indices]
    return real_images

# Assuming you have your original GAN training loop
for epoch in range(epochs):
    print('epoch:', epoch)
    for i in range(len(random_samples) // batch_size):
        print('i:', i)
        # Get a batch of real examples
        real_examples = get_real_batch(random_samples, batch_size)
        second_last_dense_layer = resnet18_model.get_layer('dense1')

        # Alternatively, you can access the second last layer by index
        # Index -2 corresponds to the second last layer
        #second_last_dense_layer = model.get_layer('dense_156')
    
        # Create a sub-model to obtain latent code
        latent_code_model = models.Model(inputs=resnet18_model.input, 
                                         outputs=second_last_dense_layer.output)
        
        # Assuming you have an input image 'input_image'
        latent_code = latent_code_model.predict(random_samples)
        print("Latent Code Shape:", latent_code.shape)
        
        from tensorflow.keras.utils import to_categorical
        one_hot_labels = to_categorical(labels1, num_classes=2)
        print("one_hot_labels  Shape:", one_hot_labels.shape)
        
        
        # Generate denoised examples using the current latent code
        denoised_examples = G.predict([latent_code, one_hot_labels])
        #denoised_images_pgd = G.predict([latent_code, one_hot_encoded]) #G1.predict([reshaped_tensor, one_hot_encoded])
        print('denoised_examples:',denoised_examples.shape)
        resnet18_model.evaluate(denoised_examples, one_hot_labels)

        # Get a batch of denoised examples
        denoised_examples_batch = get_real_batch(denoised_examples, batch_size)

        # Train the discriminator on real examples
        '''
        realLabels = self.labels[indexs]
            realTag = tf.ones(shape = (self.batch_size, ))
        fakeLabels = tf.keras.utils.to_categorical(np.random.choice(range(3), size = (self.batch_size)),
                                                      num_classes = 3)
            fakeImages = tf.squeeze(G.predict([noize, fakeLabels], verbose = 0))
            #fakeImages = (G.predict([noize, fakeLabels], verbose = 0))
            #fakeImages, _, _ = G.predict([noize, fakeLabels], verbose=0)
            fakeTag = tf.zeros(shape = (self.batch_size, ))
            allImages = np.vstack([realImages, fakeImages])
            allLabels = np.vstack([realLabels, fakeLabels])
            allTags = np.hstack([realTag, fakeTag])
            _, dlossTag, dlossLabels = D.train_on_batch(allImages, [allTags, allLabels])
        '''
        #fakeLabels = one_hot_labels
        fakeLabels = get_real_batch(one_hot_labels, batch_size)
        #fakeTag = tf.zeros(shape = (batch_size, ))
        fakeTag = tf.concat([tf.zeros((batch_size, 1)), tf.zeros((batch_size, 1)),
                                               tf.zeros((batch_size, 1))
                                                      ], axis=1)
        realLabels = get_real_batch(one_hot_labels, batch_size)
        
        #realTag = tf.ones(shape = (batch_size, ))
        realTag = tf.concat([tf.ones((batch_size, 1)), tf.ones((batch_size, 1)),
                                               tf.ones((batch_size, 1))
                                                      ], axis=1)
        
        d_loss_real = D.train_on_batch(real_examples,[realTag, realLabels])

        # Train the discriminator on denoised examples
        d_loss_fake = D.train_on_batch(denoised_examples_batch, [fakeTag, fakeLabels])

        # Calculate the total discriminator loss
        d_loss_real_value = d_loss_real[0]
        d_loss_fake_value = d_loss_fake[0]

        # Calculate the total discriminator loss
        d_loss = 0.5 * (d_loss_real_value + d_loss_fake_value)

        # Train the generator to fool the discriminator
        latent_code_examples = get_real_batch(latent_code, batch_size)
        one_hot_labels_examples = get_real_batch(one_hot_labels, batch_size)
        #_, glossTag, glossLabels = GAN.train_on_batch([noize, fakeLabels], [realTag, fakeLabels])
        g_loss = GAN.train_on_batch([latent_code_examples, one_hot_labels_examples], 
                                    [realTag, one_hot_labels_examples])

    print(f"Epoch {epoch + 1}, D Loss: {d_loss}, G Loss: {g_loss}")

# After the training loop, you can use the updated generator for denoising
denoised_image = G.predict([latent_code, one_hot_labels])
print("Denoised Image Shape:", denoised_image.shape)


In [ ]:
from tensorflow.keras.utils import to_categorical
one_hot_labels = to_categorical(labels1, num_classes=2)

resnet18_model.evaluate(random_samples, one_hot_labels)

second_last_dense_layer = resnet18_model.get_layer('dense1')

latent_code_model = models.Model(inputs=resnet18_model.input, outputs=second_last_dense_layer.output)

# Assuming you have an input image 'input_image'
latent_code = latent_code_model.predict(images_test)
print("Latent Code Shape:", latent_code.shape)

one_hot_labels_test = to_categorical(labels_test, num_classes=2)
denoised_images_pgd = G.predict([latent_code, one_hot_labels_test]) #G1.predict([reshaped_tensor, one_hot_encoded])
print(denoised_images_pgd.shape)
resnet18_model.evaluate(denoised_images_pgd, one_hot_labels_test)

In [ ]:
resnet18_model.evaluate(denoised_image, one_hot_labels)

In [ ]:
resnet18_model.evaluate(denoised_images_pgd, one_hot_labels)

In [ ]:
denoised_image = G.predict([latent_code, one_hot_labels])

In [ ]:
model = resnet18_model

In [ ]:
import pandas as pd
reshaped_tensor = tf.reduce_mean(images_test, axis=(2, 3))
print(reshaped_tensor.shape)

#one_hot_encoded = pd.get_dummies(labels_test1)
one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
print(one_hot_encoded.shape)
model.evaluate(images_test, one_hot_encoded)
print(reshaped_tensor.shape, one_hot_encoded.shape)

#denoised_features_fgsm = G.predict([reshaped_tensor, one_hot_encoded])
#print(denoised_features_fgsm.shape)

denoised_images_pgd = G.predict([reshaped_tensor, one_hot_encoded]) #G1.predict([reshaped_tensor, one_hot_encoded])
print(denoised_images_pgd.shape)
model.evaluate(denoised_images_pgd, one_hot_encoded)

In [ ]:
import numpy as np
y_test_one_hot = to_categorical(labels_test, num_classes=2)
model.evaluate(images_test, y_test_one_hot)
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Split the evaluation into batches
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

adversarial_examples = []

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

    # Generate adversarial examples for the current batch
    adv_batch = momentum_iterative_method(
        model_fn=model,
        x=images_test[start_idx:end_idx],
        eps=0.1,
        eps_iter=0.01,
        nb_iter=10,
        norm=np.inf,
        clip_min=None,
        clip_max=None,
        y=labels_test[start_idx:end_idx],
        targeted=False,
        decay_factor=1.0,
        sanity_checks=True,
    )

    adversarial_examples.append(adv_batch)

# Concatenate the adversarial examples from all batches
adversarial_examples = np.concatenate(adversarial_examples, axis=0)


# Evaluate the model on the concatenated adversarial examples
model.evaluate(adversarial_examples, y_test_one_hot)


In [ ]:
import os
import zipfile
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent

# Assuming you have defined 'model', 'images_test', 'labels_test', 'num_batches', 'num_samples', 'y_test_one_hot'

# Choose a batch size
batch_size = 20


output_dir = '/working/'

# Create a directory to store the adversarial examples and predictions
adversarial_examples_dir = os.path.join(output_dir, 'adversarial_examples')
os.makedirs(adversarial_examples_dir, exist_ok=True)

# Create a directory to store the predictions
predictions_dir = os.path.join(output_dir, 'predictions')
os.makedirs(predictions_dir, exist_ok=True)

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = projected_gradient_descent(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon / 4,
            nb_iter=10,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Save adversarial examples to numpy file
    np_file_path = os.path.join(adversarial_examples_dir, f'adversarial_examples_epsilon_{epsilon}.npy')
    np.save(np_file_path, adversarial_examples)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    })

    # Save predictions to numpy file
    np.save(os.path.join(predictions_dir, f'predictions_epsilon_{epsilon}.npy'), y_pred.numpy())

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    print('-' * 50)

# Zip the numpy files (adversarial examples)
with zipfile.ZipFile(os.path.join(output_dir, 'adversarial_examples_pgd_100.zip'), 'w') as zipf:
    for epsilon in epsilon_values:
        np_file_path = os.path.join(adversarial_examples_dir, f'adversarial_examples_epsilon_{epsilon}.npy')
        zipf.write(np_file_path, os.path.basename(np_file_path))

# Zip the numpy files (predictions)
with zipfile.ZipFile(os.path.join(output_dir, 'predictions_pgd_100.zip'), 'w') as zipf:
    for epsilon in epsilon_values:
        np_file_path = os.path.join(predictions_dir, f'predictions_epsilon_{epsilon}.npy')
        zipf.write(np_file_path, os.path.basename(np_file_path))

# Remove the numpy files (adversarial examples)
for epsilon in epsilon_values:
    np_file_path = os.path.join(adversarial_examples_dir, f'adversarial_examples_epsilon_{epsilon}.npy')
    os.remove(np_file_path)

# Remove the adversarial examples directory
os.rmdir(adversarial_examples_dir)

# Remove the numpy files (predictions)
for epsilon in epsilon_values:
    np_file_path = os.path.join(predictions_dir, f'predictions_epsilon_{epsilon}.npy')
    os.remove(np_file_path)

# Remove the predictions directory
os.rmdir(predictions_dir)


In [ ]:
import os
import zipfile
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method
# Assuming you have defined 'model', 'images_test', 'labels_test', 'num_batches', 'num_samples', 'y_test_one_hot'

# Choose a batch size
batch_size = 20


output_dir = '/working/'

# Create a directory to store the adversarial examples and predictions
adversarial_examples_dir = os.path.join(output_dir, 'adversarial_examples')
os.makedirs(adversarial_examples_dir, exist_ok=True)

# Create a directory to store the predictions
predictions_dir = os.path.join(output_dir, 'predictions')
os.makedirs(predictions_dir, exist_ok=True)

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=10,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Save adversarial examples to numpy file
    np_file_path = os.path.join(adversarial_examples_dir, f'adversarial_examples_epsilon_{epsilon}.npy')
    np.save(np_file_path, adversarial_examples)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    })

    # Save predictions to numpy file
    np.save(os.path.join(predictions_dir, f'predictions_epsilon_{epsilon}.npy'), y_pred.numpy())

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    print('-' * 50)

# Zip the numpy files (adversarial examples)
with zipfile.ZipFile(os.path.join(output_dir, 'adversarial_examples_mim_10.zip'), 'w') as zipf:
    for epsilon in epsilon_values:
        np_file_path = os.path.join(adversarial_examples_dir, f'adversarial_examples_epsilon_{epsilon}.npy')
        zipf.write(np_file_path, os.path.basename(np_file_path))

# Zip the numpy files (predictions)
with zipfile.ZipFile(os.path.join(output_dir, 'predictions_mim_10.zip'), 'w') as zipf:
    for epsilon in epsilon_values:
        np_file_path = os.path.join(predictions_dir, f'predictions_epsilon_{epsilon}.npy')
        zipf.write(np_file_path, os.path.basename(np_file_path))

# Remove the numpy files (adversarial examples)
for epsilon in epsilon_values:
    np_file_path = os.path.join(adversarial_examples_dir, f'adversarial_examples_epsilon_{epsilon}.npy')
    os.remove(np_file_path)

# Remove the adversarial examples directory
os.rmdir(adversarial_examples_dir)

# Remove the numpy files (predictions)
for epsilon in epsilon_values:
    np_file_path = os.path.join(predictions_dir, f'predictions_epsilon_{epsilon}.npy')
    os.remove(np_file_path)

# Remove the predictions directory
os.rmdir(predictions_dir)


In [ ]:
import os
import zipfile
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Assuming you have defined 'model', 'images_test', 'labels_test', 'num_batches', 'num_samples', 'y_test_one_hot'

# Choose a batch size
batch_size = 20


output_dir = '/working/'

# Create a directory to store the adversarial examples and predictions
adversarial_examples_dir = os.path.join(output_dir, 'adversarial_examples')
os.makedirs(adversarial_examples_dir, exist_ok=True)

# Create a directory to store the predictions
predictions_dir = os.path.join(output_dir, 'predictions')
os.makedirs(predictions_dir, exist_ok=True)

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=10,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )


        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Save adversarial examples to numpy file
    np_file_path = os.path.join(adversarial_examples_dir, f'adversarial_examples_epsilon_{epsilon}.npy')
    np.save(np_file_path, adversarial_examples)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    })

    # Save predictions to numpy file
    np.save(os.path.join(predictions_dir, f'predictions_epsilon_{epsilon}.npy'), y_pred.numpy())

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    print('-' * 50)

# Zip the numpy files (adversarial examples)
with zipfile.ZipFile(os.path.join(output_dir, 'adversarial_examples_bim_10.zip'), 'w') as zipf:
    for epsilon in epsilon_values:
        np_file_path = os.path.join(adversarial_examples_dir, f'adversarial_examples_epsilon_{epsilon}.npy')
        zipf.write(np_file_path, os.path.basename(np_file_path))

# Zip the numpy files (predictions)
with zipfile.ZipFile(os.path.join(output_dir, 'predictions_bim_10.zip'), 'w') as zipf:
    for epsilon in epsilon_values:
        np_file_path = os.path.join(predictions_dir, f'predictions_epsilon_{epsilon}.npy')
        zipf.write(np_file_path, os.path.basename(np_file_path))

# Remove the numpy files (adversarial examples)
for epsilon in epsilon_values:
    np_file_path = os.path.join(adversarial_examples_dir, f'adversarial_examples_epsilon_{epsilon}.npy')
    os.remove(np_file_path)

# Remove the adversarial examples directory
os.rmdir(adversarial_examples_dir)

# Remove the numpy files (predictions)
for epsilon in epsilon_values:
    np_file_path = os.path.join(predictions_dir, f'predictions_epsilon_{epsilon}.npy')
    os.remove(np_file_path)

# Remove the predictions directory
os.rmdir(predictions_dir)


In [ ]:
import numpy as np
import shutil
### for PGD-100

epsilon_values = [0, 0.01,
                 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1
                 ]

results = {}  # Dictionary to store evaluation results for different epsilon values

# Load PGD adversarial examples and concatenate them
adversarial_examples = []

for epsilon in epsilon_values:
    adversarial_x_pgd = np.load(f'/working/adversarial_examples_pgd_100/adversarial_examples_pgd_10/adversarial_examples_epsilon_{epsilon}.npy')
    #adversarial_x_pgd = np.squeeze(adversarial_x_pgd, axis=0)
    print(adversarial_x_pgd.shape)
    import pandas as pd
    #reshaped_tensor = tf.reduce_mean(adversarial_x_pgd, axis=(2, 3))
    #print(reshaped_tensor.shape)
    latent_code_model = models.Model(inputs=resnet18_model.input, outputs=second_last_dense_layer.output)
    latent_code = latent_code_model.predict(adversarial_x_pgd)
    print("Latent Code Shape:", latent_code.shape)
    
    #one_hot_encoded = pd.get_dummies(labels_test1)
    one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
    print(one_hot_encoded.shape)
    
    print(latent_code.shape, one_hot_encoded.shape)
    
    #denoised_features_fgsm = G.predict([reshaped_tensor, one_hot_encoded])
    #print(denoised_features_fgsm.shape)
    
    denoised_images_pgd = G.predict([latent_code, one_hot_encoded]) #G1.predict([reshaped_tensor, one_hot_encoded])
    print(denoised_images_pgd.shape)
    

    print()
    y_pred = tf.squeeze(model.predict(denoised_images_pgd))
    y_pred = y_pred >= 0.5
    y_pred = np.array(y_pred, dtype='int32')
    
    accuracy = accuracy_score(y_pred, one_hot_encoded) * 100
    precision = precision_score(y_pred, one_hot_encoded, average='macro') * 100
    recall = recall_score(y_pred, one_hot_encoded, average='macro') * 100
    f1 = f1_score(y_pred, one_hot_encoded, average='macro') * 100

    results[epsilon] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print('Epsilon value:', epsilon)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('-' * 50)
    
    

    
    

    #adversarial_examples.append(adversarial_x_pgd)


In [ ]:
#pgd100
import numpy as np
import shutil
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import zipfile

# Assuming latent_code_model, G, and model are defined elsewhere in your code

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
results = {}  # Dictionary to store evaluation results for different epsilon values

# Output directory for saving denoised examples and predictions
output_directory = '/working/denoised_results_mim_10/'

# Create the output directory
shutil.rmtree(output_directory, ignore_errors=True)
os.makedirs(output_directory, exist_ok=True)

for epsilon in epsilon_values:
    adversarial_x_pgd = np.load(f'/input/adversarial-examples-chest-xray/adversarial_examples_pgd_10/adversarial_examples_epsilon_{epsilon}.npy')

    # reshaped_tensor = tf.reduce_mean(adversarial_x_pgd, axis=(2, 3))
    latent_code = latent_code_model.predict(adversarial_x_pgd)
    print('latent_code', latent_code.shape)
    one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
    print('one_hot_encoded', one_hot_encoded.shape)
    denoised_images_pgd = G.predict([latent_code, one_hot_encoded])

    y_pred = tf.squeeze(resnet18_model.predict(denoised_images_pgd))
    y_pred = y_pred >= 0.5
    y_pred = np.array(y_pred, dtype='int32')

    # Save denoised examples and predictions to numpy files
    np.save(os.path.join(output_directory, f'denoised_examples_epsilon_{epsilon}.npy'), denoised_images_pgd)
    np.save(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'), y_pred)

    accuracy = accuracy_score(y_pred, one_hot_encoded) * 100
    precision = precision_score(y_pred, one_hot_encoded, average='macro') * 100
    recall = recall_score(y_pred, one_hot_encoded, average='macro') * 100
    f1 = f1_score(y_pred, one_hot_encoded, average='macro') * 100

    results[epsilon] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print('Epsilon value:', epsilon)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('-' * 50)

# Zip the output directory
with zipfile.ZipFile('/working/denoised_results_pgd_100_adversarial_examples.zip', 'w') as zipf:
    for root, _, files in os.walk(output_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_directory))

# Create a new output directory for predictions
output_predictions_directory = '/working/denoised_results_mim_10_predictions/'

# Create the output directory for predictions
os.makedirs(output_predictions_directory, exist_ok=True)

# Move prediction files to the new directory
for epsilon in epsilon_values:
    shutil.move(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'),
                os.path.join(output_predictions_directory, f'predictions_epsilon_{epsilon}.npy'))

# Zip the prediction directory
with zipfile.ZipFile('/working/denoised_results_pgd_100_predictions.zip', 'w') as zipf:
    for root, _, files in os.walk(output_predictions_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_predictions_directory))


In [ ]:
#pgd100
import numpy as np
import shutil
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import zipfile

# Assuming latent_code_model, G, and model are defined elsewhere in your code

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
results = {}  # Dictionary to store evaluation results for different epsilon values

# Output directory for saving denoised examples and predictions
output_directory = '/working/denoised_results_mim_10/'

# Create the output directory
shutil.rmtree(output_directory, ignore_errors=True)
os.makedirs(output_directory, exist_ok=True)

for epsilon in epsilon_values:
    adversarial_x_pgd = np.load(f'/input/adversarial-examples-chest-xray/adversarial_examples_bim/adversarial_examples_epsilon_{epsilon}.npy')

    # reshaped_tensor = tf.reduce_mean(adversarial_x_pgd, axis=(2, 3))
    latent_code = latent_code_model.predict(adversarial_x_pgd)
    print('latent_code', latent_code.shape)
    one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
    print('one_hot_encoded', one_hot_encoded.shape)
    denoised_images_pgd = G.predict([latent_code, one_hot_encoded])

    y_pred = tf.squeeze(resnet18_model.predict(denoised_images_pgd))
    y_pred = y_pred >= 0.5
    y_pred = np.array(y_pred, dtype='int32')

    # Save denoised examples and predictions to numpy files
    np.save(os.path.join(output_directory, f'denoised_examples_epsilon_{epsilon}.npy'), denoised_images_pgd)
    np.save(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'), y_pred)

    accuracy = accuracy_score(y_pred, one_hot_encoded) * 100
    precision = precision_score(y_pred, one_hot_encoded, average='macro') * 100
    recall = recall_score(y_pred, one_hot_encoded, average='macro') * 100
    f1 = f1_score(y_pred, one_hot_encoded, average='macro') * 100

    results[epsilon] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print('Epsilon value:', epsilon)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('-' * 50)

# Zip the output directory
with zipfile.ZipFile('/working/denoised_results_bim_100_adversarial_examples.zip', 'w') as zipf:
    for root, _, files in os.walk(output_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_directory))

# Create a new output directory for predictions
output_predictions_directory = '/working/denoised_results_mim_10_predictions/'

# Create the output directory for predictions
os.makedirs(output_predictions_directory, exist_ok=True)

# Move prediction files to the new directory
for epsilon in epsilon_values:
    shutil.move(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'),
                os.path.join(output_predictions_directory, f'predictions_epsilon_{epsilon}.npy'))

# Zip the prediction directory
with zipfile.ZipFile('/working/denoised_results_bim_100_predictions.zip', 'w') as zipf:
    for root, _, files in os.walk(output_predictions_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_predictions_directory))


In [ ]:
#pgd10
import numpy as np
import shutil
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import zipfile

# Assuming latent_code_model, G, and model are defined elsewhere in your code

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
results = {}  # Dictionary to store evaluation results for different epsilon values

# Output directory for saving denoised examples and predictions
output_directory = '/working/denoised_results_mim_10/'

# Create the output directory
shutil.rmtree(output_directory, ignore_errors=True)
os.makedirs(output_directory, exist_ok=True)

for epsilon in epsilon_values:
    adversarial_x_pgd = np.load(f'/input/recent-adversarial-examples-covid-cxr-resnet/adversarial_examples_pgd_10/adversarial_examples_epsilon_{epsilon}.npy')

    # reshaped_tensor = tf.reduce_mean(adversarial_x_pgd, axis=(2, 3))
    latent_code = latent_code_model.predict(adversarial_x_pgd)
    print('latent_code', latent_code.shape)
    one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
    print('one_hot_encoded', one_hot_encoded.shape)
    denoised_images_pgd = G.predict([latent_code, one_hot_encoded])

    y_pred = tf.squeeze(resnet18_model.predict(denoised_images_pgd))
    y_pred = y_pred >= 0.5
    y_pred = np.array(y_pred, dtype='int32')

    # Save denoised examples and predictions to numpy files
    np.save(os.path.join(output_directory, f'denoised_examples_epsilon_{epsilon}.npy'), denoised_images_pgd)
    np.save(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'), y_pred)

    accuracy = accuracy_score(y_pred, one_hot_encoded) * 100
    precision = precision_score(y_pred, one_hot_encoded, average='macro') * 100
    recall = recall_score(y_pred, one_hot_encoded, average='macro') * 100
    f1 = f1_score(y_pred, one_hot_encoded, average='macro') * 100

    results[epsilon] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print('Epsilon value:', epsilon)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('-' * 50)

# Zip the output directory
with zipfile.ZipFile('/working/denoised_results_pgd_10_adversarial_examples.zip', 'w') as zipf:
    for root, _, files in os.walk(output_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_directory))

# Create a new output directory for predictions
output_predictions_directory = '/working/denoised_results_mim_10_predictions/'

# Create the output directory for predictions
os.makedirs(output_predictions_directory, exist_ok=True)

# Move prediction files to the new directory
for epsilon in epsilon_values:
    shutil.move(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'),
                os.path.join(output_predictions_directory, f'predictions_epsilon_{epsilon}.npy'))

# Zip the prediction directory
with zipfile.ZipFile('/working/denoised_results_pgd_10_predictions.zip', 'w') as zipf:
    for root, _, files in os.walk(output_predictions_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_predictions_directory))


In [ ]:
#bim10
import numpy as np
import shutil
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import zipfile

# Assuming latent_code_model, G, and model are defined elsewhere in your code

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
results = {}  # Dictionary to store evaluation results for different epsilon values

# Output directory for saving denoised examples and predictions
output_directory = '/working/denoised_results_mim_10/'

# Create the output directory
shutil.rmtree(output_directory, ignore_errors=True)
os.makedirs(output_directory, exist_ok=True)

for epsilon in epsilon_values:
    adversarial_x_pgd = np.load(f'/input/adversarial-examples-chest-xray/adversarial_examples_mim/adversarial_examples_epsilon_{epsilon}.npy')

    # reshaped_tensor = tf.reduce_mean(adversarial_x_pgd, axis=(2, 3))
    latent_code = latent_code_model.predict(adversarial_x_pgd)
    print('latent_code', latent_code.shape)
    one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
    print('one_hot_encoded', one_hot_encoded.shape)
    denoised_images_pgd = G.predict([latent_code, one_hot_encoded])

    y_pred = tf.squeeze(resnet18_model.predict(denoised_images_pgd))
    y_pred = y_pred >= 0.5
    y_pred = np.array(y_pred, dtype='int32')

    # Save denoised examples and predictions to numpy files
    np.save(os.path.join(output_directory, f'denoised_examples_epsilon_{epsilon}.npy'), denoised_images_pgd)
    np.save(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'), y_pred)

    accuracy = accuracy_score(y_pred, one_hot_encoded) * 100
    precision = precision_score(y_pred, one_hot_encoded, average='macro') * 100
    recall = recall_score(y_pred, one_hot_encoded, average='macro') * 100
    f1 = f1_score(y_pred, one_hot_encoded, average='macro') * 100

    results[epsilon] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print('Epsilon value:', epsilon)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('-' * 50)

# Zip the output directory
with zipfile.ZipFile('/working/denoised_results_bim_10_adversarial_examples.zip', 'w') as zipf:
    for root, _, files in os.walk(output_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_directory))

# Create a new output directory for predictions
output_predictions_directory = '/working/denoised_results_mim_10_predictions/'

# Create the output directory for predictions
os.makedirs(output_predictions_directory, exist_ok=True)

# Move prediction files to the new directory
for epsilon in epsilon_values:
    shutil.move(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'),
                os.path.join(output_predictions_directory, f'predictions_epsilon_{epsilon}.npy'))

# Zip the prediction directory
with zipfile.ZipFile('/working/denoised_results_bim_10_predictions.zip', 'w') as zipf:
    for root, _, files in os.walk(output_predictions_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_predictions_directory))


In [ ]:
#pgd4
import numpy as np
import shutil
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import zipfile

# Assuming latent_code_model, G, and model are defined elsewhere in your code

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
results = {}  # Dictionary to store evaluation results for different epsilon values

# Output directory for saving denoised examples and predictions
output_directory = '/working/denoised_results_mim_10/'

# Create the output directory
shutil.rmtree(output_directory, ignore_errors=True)
os.makedirs(output_directory, exist_ok=True)

for epsilon in epsilon_values:
    adversarial_x_pgd = np.load(f'/input/recent-adversarial-examples-covid-cxr-resnet/adversarial_examples_pgd_4/adversarial_examples_epsilon_{epsilon}.npy')

    # reshaped_tensor = tf.reduce_mean(adversarial_x_pgd, axis=(2, 3))
    latent_code = latent_code_model.predict(adversarial_x_pgd)
    print('latent_code', latent_code.shape)
    one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
    print('one_hot_encoded', one_hot_encoded.shape)
    denoised_images_pgd = G.predict([latent_code, one_hot_encoded])

    y_pred = tf.squeeze(resnet18_model.predict(denoised_images_pgd))
    y_pred = y_pred >= 0.5
    y_pred = np.array(y_pred, dtype='int32')

    # Save denoised examples and predictions to numpy files
    np.save(os.path.join(output_directory, f'denoised_examples_epsilon_{epsilon}.npy'), denoised_images_pgd)
    np.save(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'), y_pred)

    accuracy = accuracy_score(y_pred, one_hot_encoded) * 100
    precision = precision_score(y_pred, one_hot_encoded, average='macro') * 100
    recall = recall_score(y_pred, one_hot_encoded, average='macro') * 100
    f1 = f1_score(y_pred, one_hot_encoded, average='macro') * 100

    results[epsilon] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print('Epsilon value:', epsilon)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('-' * 50)

# Zip the output directory
with zipfile.ZipFile('/working/denoised_results_pgd_4_adversarial_examples.zip', 'w') as zipf:
    for root, _, files in os.walk(output_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_directory))

# Create a new output directory for predictions
output_predictions_directory = '/working/denoised_results_mim_10_predictions/'

# Create the output directory for predictions
os.makedirs(output_predictions_directory, exist_ok=True)

# Move prediction files to the new directory
for epsilon in epsilon_values:
    shutil.move(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'),
                os.path.join(output_predictions_directory, f'predictions_epsilon_{epsilon}.npy'))

# Zip the prediction directory
with zipfile.ZipFile('/working/denoised_results_pgd_4_predictions.zip', 'w') as zipf:
    for root, _, files in os.walk(output_predictions_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_predictions_directory))


In [ ]:
#bim10
import numpy as np
import shutil
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import zipfile

# Assuming latent_code_model, G, and model are defined elsewhere in your code

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
results = {}  # Dictionary to store evaluation results for different epsilon values

# Output directory for saving denoised examples and predictions
output_directory = '/working/denoised_results_mim_10/'

# Create the output directory
shutil.rmtree(output_directory, ignore_errors=True)
os.makedirs(output_directory, exist_ok=True)

for epsilon in epsilon_values:
    adversarial_x_pgd = np.load(f'/input/recent-adversarial-examples-covid-cxr-resnet/adversarial_examples_bim_4/adversarial_examples_epsilon_{epsilon}.npy')

    # reshaped_tensor = tf.reduce_mean(adversarial_x_pgd, axis=(2, 3))
    latent_code = latent_code_model.predict(adversarial_x_pgd)
    print('latent_code', latent_code.shape)
    one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
    print('one_hot_encoded', one_hot_encoded.shape)
    denoised_images_pgd = G.predict([latent_code, one_hot_encoded])

    y_pred = tf.squeeze(resnet18_model.predict(denoised_images_pgd))
    y_pred = y_pred >= 0.5
    y_pred = np.array(y_pred, dtype='int32')

    # Save denoised examples and predictions to numpy files
    np.save(os.path.join(output_directory, f'denoised_examples_epsilon_{epsilon}.npy'), denoised_images_pgd)
    np.save(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'), y_pred)

    accuracy = accuracy_score(y_pred, one_hot_encoded) * 100
    precision = precision_score(y_pred, one_hot_encoded, average='macro') * 100
    recall = recall_score(y_pred, one_hot_encoded, average='macro') * 100
    f1 = f1_score(y_pred, one_hot_encoded, average='macro') * 100

    results[epsilon] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print('Epsilon value:', epsilon)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('-' * 50)

# Zip the output directory
with zipfile.ZipFile('/working/denoised_results_bim_4_adversarial_examples.zip', 'w') as zipf:
    for root, _, files in os.walk(output_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_directory))

# Create a new output directory for predictions
output_predictions_directory = '/working/denoised_results_mim_10_predictions/'

# Create the output directory for predictions
os.makedirs(output_predictions_directory, exist_ok=True)

# Move prediction files to the new directory
for epsilon in epsilon_values:
    shutil.move(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'),
                os.path.join(output_predictions_directory, f'predictions_epsilon_{epsilon}.npy'))

# Zip the prediction directory
with zipfile.ZipFile('/working/denoised_results_bim_4_predictions.zip', 'w') as zipf:
    for root, _, files in os.walk(output_predictions_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_predictions_directory))


In [ ]:
#bim10
import numpy as np
import shutil
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import zipfile

# Assuming latent_code_model, G, and model are defined elsewhere in your code

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
results = {}  # Dictionary to store evaluation results for different epsilon values

# Output directory for saving denoised examples and predictions
output_directory = '/working/denoised_results_mim_10/'

# Create the output directory
shutil.rmtree(output_directory, ignore_errors=True)
os.makedirs(output_directory, exist_ok=True)

for epsilon in epsilon_values:
    adversarial_x_pgd = np.load(f'/input/recent-adversarial-examples-covid-cxr-resnet/adversarial_examples_mim_4/adversarial_examples_epsilon_{epsilon}.npy')

    # reshaped_tensor = tf.reduce_mean(adversarial_x_pgd, axis=(2, 3))
    latent_code = latent_code_model.predict(adversarial_x_pgd)
    print('latent_code', latent_code.shape)
    one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
    print('one_hot_encoded', one_hot_encoded.shape)
    denoised_images_pgd = G.predict([latent_code, one_hot_encoded])

    y_pred = tf.squeeze(resnet18_model.predict(denoised_images_pgd))
    y_pred = y_pred >= 0.5
    y_pred = np.array(y_pred, dtype='int32')

    # Save denoised examples and predictions to numpy files
    np.save(os.path.join(output_directory, f'denoised_examples_epsilon_{epsilon}.npy'), denoised_images_pgd)
    np.save(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'), y_pred)

    accuracy = accuracy_score(y_pred, one_hot_encoded) * 100
    precision = precision_score(y_pred, one_hot_encoded, average='macro') * 100
    recall = recall_score(y_pred, one_hot_encoded, average='macro') * 100
    f1 = f1_score(y_pred, one_hot_encoded, average='macro') * 100

    results[epsilon] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print('Epsilon value:', epsilon)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('-' * 50)

# Zip the output directory
with zipfile.ZipFile('/working/denoised_results_mim_4_adversarial_examples.zip', 'w') as zipf:
    for root, _, files in os.walk(output_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_directory))

# Create a new output directory for predictions
output_predictions_directory = '/working/denoised_results_mim_10_predictions/'

# Create the output directory for predictions
os.makedirs(output_predictions_directory, exist_ok=True)

# Move prediction files to the new directory
for epsilon in epsilon_values:
    shutil.move(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'),
                os.path.join(output_predictions_directory, f'predictions_epsilon_{epsilon}.npy'))

# Zip the prediction directory
with zipfile.ZipFile('/working/denoised_results_mim_4_predictions.zip', 'w') as zipf:
    for root, _, files in os.walk(output_predictions_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_predictions_directory))


In [ ]:
#bim10
import numpy as np
import shutil
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import zipfile

# Assuming latent_code_model, G, and model are defined elsewhere in your code

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
results = {}  # Dictionary to store evaluation results for different epsilon values

# Output directory for saving denoised examples and predictions
output_directory = '/working/denoised_results_mim_10/'

# Create the output directory
shutil.rmtree(output_directory, ignore_errors=True)
os.makedirs(output_directory, exist_ok=True)

for epsilon in epsilon_values:
    adversarial_x_pgd = np.load(f'/input/recent-adversarial-examples-covid-cxr-resnet/adversarial_examples_mim_10/adversarial_examples_epsilon_{epsilon}.npy')

    # reshaped_tensor = tf.reduce_mean(adversarial_x_pgd, axis=(2, 3))
    latent_code = latent_code_model.predict(adversarial_x_pgd)
    print('latent_code', latent_code.shape)
    one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
    print('one_hot_encoded', one_hot_encoded.shape)
    denoised_images_pgd = G.predict([latent_code, one_hot_encoded])

    y_pred = tf.squeeze(resnet18_model.predict(denoised_images_pgd))
    y_pred = y_pred >= 0.5
    y_pred = np.array(y_pred, dtype='int32')

    # Save denoised examples and predictions to numpy files
    np.save(os.path.join(output_directory, f'denoised_examples_epsilon_{epsilon}.npy'), denoised_images_pgd)
    np.save(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'), y_pred)

    accuracy = accuracy_score(y_pred, one_hot_encoded) * 100
    precision = precision_score(y_pred, one_hot_encoded, average='macro') * 100
    recall = recall_score(y_pred, one_hot_encoded, average='macro') * 100
    f1 = f1_score(y_pred, one_hot_encoded, average='macro') * 100

    results[epsilon] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print('Epsilon value:', epsilon)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('-' * 50)

# Zip the output directory
with zipfile.ZipFile('/working/denoised_results_mim_10_adversarial_examples.zip', 'w') as zipf:
    for root, _, files in os.walk(output_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_directory))

# Create a new output directory for predictions
output_predictions_directory = '/working/denoised_results_mim_10_predictions/'

# Create the output directory for predictions
os.makedirs(output_predictions_directory, exist_ok=True)

# Move prediction files to the new directory
for epsilon in epsilon_values:
    shutil.move(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'),
                os.path.join(output_predictions_directory, f'predictions_epsilon_{epsilon}.npy'))

# Zip the prediction directory
with zipfile.ZipFile('/working/denoised_results_mim_10_predictions.zip', 'w') as zipf:
    for root, _, files in os.walk(output_predictions_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_predictions_directory))


In [ ]:
#bim10
import numpy as np
import shutil
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import zipfile

# Assuming latent_code_model, G, and model are defined elsewhere in your code

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
results = {}  # Dictionary to store evaluation results for different epsilon values

# Output directory for saving denoised examples and predictions
output_directory = '/working/denoised_results_mim_10/'

# Create the output directory
shutil.rmtree(output_directory, ignore_errors=True)
os.makedirs(output_directory, exist_ok=True)

for epsilon in epsilon_values:
    adversarial_x_pgd = np.load(f'/input/recent-adversarial-examples-covid-cxr-resnet/adversarial_examples_mim_100/adversarial_examples_epsilon_{epsilon}.npy')

    # reshaped_tensor = tf.reduce_mean(adversarial_x_pgd, axis=(2, 3))
    latent_code = latent_code_model.predict(adversarial_x_pgd)
    print('latent_code', latent_code.shape)
    one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
    print('one_hot_encoded', one_hot_encoded.shape)
    denoised_images_pgd = G.predict([latent_code, one_hot_encoded])

    y_pred = tf.squeeze(resnet18_model.predict(denoised_images_pgd))
    y_pred = y_pred >= 0.5
    y_pred = np.array(y_pred, dtype='int32')

    # Save denoised examples and predictions to numpy files
    np.save(os.path.join(output_directory, f'denoised_examples_epsilon_{epsilon}.npy'), denoised_images_pgd)
    np.save(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'), y_pred)

    accuracy = accuracy_score(y_pred, one_hot_encoded) * 100
    precision = precision_score(y_pred, one_hot_encoded, average='macro') * 100
    recall = recall_score(y_pred, one_hot_encoded, average='macro') * 100
    f1 = f1_score(y_pred, one_hot_encoded, average='macro') * 100

    results[epsilon] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print('Epsilon value:', epsilon)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('-' * 50)

# Zip the output directory
with zipfile.ZipFile('/working/denoised_results_mim_100_adversarial_examples.zip', 'w') as zipf:
    for root, _, files in os.walk(output_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_directory))

# Create a new output directory for predictions
output_predictions_directory = '/working/denoised_results_mim_10_predictions/'

# Create the output directory for predictions
os.makedirs(output_predictions_directory, exist_ok=True)

# Move prediction files to the new directory
for epsilon in epsilon_values:
    shutil.move(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'),
                os.path.join(output_predictions_directory, f'predictions_epsilon_{epsilon}.npy'))

# Zip the prediction directory
with zipfile.ZipFile('/working/denoised_results_mim_100_predictions.zip', 'w') as zipf:
    for root, _, files in os.walk(output_predictions_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_predictions_directory))


In [ ]:
#bim10
import numpy as np
import shutil
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import zipfile

# Assuming latent_code_model, G, and model are defined elsewhere in your code

epsilon_values = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
results = {}  # Dictionary to store evaluation results for different epsilon values

# Output directory for saving denoised examples and predictions
output_directory = '/working/denoised_results_mim_10/'

# Create the output directory
shutil.rmtree(output_directory, ignore_errors=True)
os.makedirs(output_directory, exist_ok=True)

for epsilon in epsilon_values:
    adversarial_x_pgd = np.load(f'/input/recent-adversarial-examples-covid-cxr-resnet/adversarial_examples_fgsm/adversarial_examples_epsilon_{epsilon}.npy')

    # reshaped_tensor = tf.reduce_mean(adversarial_x_pgd, axis=(2, 3))
    latent_code = latent_code_model.predict(adversarial_x_pgd)
    print('latent_code', latent_code.shape)
    one_hot_encoded = pd.get_dummies(tf.squeeze(labels_test))
    print('one_hot_encoded', one_hot_encoded.shape)
    denoised_images_pgd = G.predict([latent_code, one_hot_encoded])

    y_pred = tf.squeeze(resnet18_model.predict(denoised_images_pgd))
    y_pred = y_pred >= 0.5
    y_pred = np.array(y_pred, dtype='int32')

    # Save denoised examples and predictions to numpy files
    np.save(os.path.join(output_directory, f'denoised_examples_epsilon_{epsilon}.npy'), denoised_images_pgd)
    np.save(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'), y_pred)

    accuracy = accuracy_score(y_pred, one_hot_encoded) * 100
    precision = precision_score(y_pred, one_hot_encoded, average='macro') * 100
    recall = recall_score(y_pred, one_hot_encoded, average='macro') * 100
    f1 = f1_score(y_pred, one_hot_encoded, average='macro') * 100

    results[epsilon] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print('Epsilon value:', epsilon)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('-' * 50)

# Zip the output directory
with zipfile.ZipFile('/working/denoised_results_fgsm_adversarial_examples.zip', 'w') as zipf:
    for root, _, files in os.walk(output_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_directory))

# Create a new output directory for predictions
output_predictions_directory = '/working/denoised_results_mim_10_predictions/'

# Create the output directory for predictions
os.makedirs(output_predictions_directory, exist_ok=True)

# Move prediction files to the new directory
for epsilon in epsilon_values:
    shutil.move(os.path.join(output_directory, f'predictions_epsilon_{epsilon}.npy'),
                os.path.join(output_predictions_directory, f'predictions_epsilon_{epsilon}.npy'))

# Zip the prediction directory
with zipfile.ZipFile('/working/denoised_results_fgsm_predictions.zip', 'w') as zipf:
    for root, _, files in os.walk(output_predictions_directory):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=os.path.relpath(os.path.join(root, file), output_predictions_directory))


In [ ]:
adversarial_x_bim = np.load(f'/input/adversarial-examples-chest-xray-by-resnet/adversarial_examples_bim_10/adversarial_examples_epsilon_0.03.npy')
pred_x_bim = np.load(f'/input/adversarial-examples-chest-xray-by-resnet/predictions_bim_10/predictions_epsilon_0.03.npy')
adversarial_x_bim.shape, pred_x_bim.shape

In [ ]:
adversarial_x_mim = np.load(f'/input/adversarial-examples-chest-xray-by-resnet/adversarial_examples_mim_10/adversarial_examples_epsilon_0.03.npy')
pred_x_mim = np.load(f'/input/adversarial-examples-chest-xray-by-resnet/predictions_mim_10/predictions_epsilon_0.03.npy')
adversarial_x_mim.shape, pred_x_mim.shape

In [ ]:
adversarial_x_pgd = np.load(f'/input/adversarial-examples-chest-xray-by-resnet/adversarial_examples_pgd_100/adversarial_examples_epsilon_0.03.npy')
pred_x_pgd = np.load(f'/input/adversarial-examples-chest-xray-by-resnet/predictions_pgd_100/predictions_epsilon_0.03.npy')
adversarial_x_pgd.shape, pred_x_pgd.shape

In [ ]:
denoised_x_pgd = np.load(f'/input/latest-denoised-images-chest-xray-resnet/denoised_results_pgd_100_adversarial_examples (1)/denoised_examples_epsilon_0.03.npy')
denoised_pred_x_pgd = np.load(f'/input/latest-denoised-images-chest-xray-resnet/denoised_results_pgd_100_predictions/predictions_epsilon_0.03.npy')
denoised_x_pgd.shape, denoised_pred_x_pgd.shape

In [ ]:
denoised_x_bim = np.load(f'/input/latest-denoised-images-chest-xray-resnet/denoised_results_bim_100_adversarial_examples/denoised_examples_epsilon_0.03.npy')
denoised_pred_x_bim = np.load(f'/input/latest-denoised-images-chest-xray-resnet/denoised_results_bim_100_predictions (1)/predictions_epsilon_0.03.npy')
denoised_x_bim.shape, denoised_pred_x_bim.shape

In [ ]:
denoised_x_mim = np.load(f'/input/latest-denoised-images-chest-xray-resnet/denoised_results_bim_10_adversarial_examples/denoised_examples_epsilon_0.03.npy')
denoised_pred_x_mim = np.load(f'/input/latest-denoised-images-chest-xray-resnet/denoised_results_bim_10_predictions/predictions_epsilon_0.03.npy')
denoised_x_mim.shape, denoised_pred_x_mim.shape

In [ ]:
import random
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from skimage import img_as_ubyte

# Set the figure size
fig = plt.figure(figsize=(5, 5))  # Increase the overall figure size

# Number of random images to select
num_images_to_plot = 3

# Generate random indices for selecting images
random_indices = random.sample(range(len(images_test)), num_images_to_plot)

# Create a GridSpec with three columns (for the original, denoised, and adversarial images)
# Increase width_ratios to make each subplot larger
gs = GridSpec(num_images_to_plot, 3)

for i, index in enumerate(random_indices):
    # Original image
    original_img = images_test[index]
    original_prediction_label = labels_test[index]

    # adversarial_img image 
    adversarial_img_pgd = adversarial_x_pgd[index]
    pred_x_pgd1 = np.argmax(pred_x_pgd, axis=1)
    pred_x_pgd2 = pred_x_pgd1[index]


    # denoised_img image
    denoised_img_pgd = denoised_x_pgd[index]
    denoised_pred_x_pgd1 = np.argmax(denoised_pred_x_pgd, axis=1)
    denoised_pred_x_pgd2 = denoised_pred_x_pgd1[index]


    # Convert images to NumPy arrays with compatible data type
    #original_img_uint8 = img_as_ubyte(original_img)
    #denoised_img_uint8 = img_as_ubyte(denoised_img)
    #adversarial_img_uint8 = img_as_ubyte(adversarial_img)

    # Create subplots within the GridSpec with aspect ratio matching the images
    ax0 = plt.subplot(gs[i, 0])
    ax1 = plt.subplot(gs[i, 1])
    ax2 = plt.subplot(gs[i, 2])

    # Plot the original image on the left
    ax0.imshow(original_img, cmap='gray')  # Assuming grayscale images, change cmap as needed
    #ax0.set_title('Original')
    ax0.set_title(f'Original\nPrediction: {original_prediction_label}') 

    # Plot the adversarial image in the middle

    ax1.imshow(adversarial_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    #ax1.set_title('Adversarial')
    ax1.set_title(f'Adversarial_PGD\nPrediction: {pred_x_pgd2}') 

    ax2.imshow(denoised_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    #ax2.set_title('Denoised')
    ax2.set_title(f'Clean_PGD\nPrediction: {denoised_pred_x_pgd2}') 

    # Adjust subplot spacing
    plt.tight_layout()

    # Save the plot with DPI=1024 as PNG file
    #plt.savefig('qualitative_analysis_images_chest_xray_pneu_3.jpg', dpi=1024)

    # Show the plot
    plt.show()


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from skimage import img_as_ubyte

# Set the figure size
fig = plt.figure(figsize=(8, 128))  # Increase the overall figure size

# Number of random images to select
num_images_to_plot = 64

# Generate random indices for selecting images
random_indices = random.sample(range(len(images_test)), num_images_to_plot)

# Create a GridSpec with three columns (for the original, denoised, and adversarial images)
# Increase width_ratios to make each subplot larger
gs = GridSpec(num_images_to_plot, 3, width_ratios=[1, 1, 1])

for i, index in enumerate(random_indices):
    # Original image
    original_img = images_test[index]
    original_prediction_label = labels_test[index]

    # Adversarial image 
    adversarial_img_pgd = adversarial_x_pgd[index]
    pred_x_pgd1 = np.argmax(pred_x_pgd, axis=1)
    pred_x_pgd2 = pred_x_pgd1[index]

    # Denoised image
    denoised_img_pgd = denoised_x_pgd[index]
    denoised_pred_x_pgd1 = np.argmax(denoised_pred_x_pgd, axis=1)
    denoised_pred_x_pgd2 = denoised_pred_x_pgd1[index]

    # Create subplots within the GridSpec with aspect ratio matching the images
    ax0 = plt.subplot(gs[i, 0])
    ax1 = plt.subplot(gs[i, 1])
    ax2 = plt.subplot(gs[i, 2])

    # Plot the original image on the left
    ax0.imshow(original_img, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax0.set_title(f'Original\nPrediction: {original_prediction_label}') 

    # Plot the adversarial image in the middle
    ax1.imshow(adversarial_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax1.set_title(f'PGD\nPrediction: {pred_x_pgd2}') 

    # Plot the denoised image on the right
    ax2.imshow(denoised_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax2.set_title(f'Purified\nPrediction: {denoised_pred_x_pgd2}') 

# Adjust subplot spacing
plt.tight_layout()
#plt.savefig('qualitative_analysis_images_chest_xray_pneu.jpg', dpi=1024)
plt.savefig('qualitative_analysis_images_chest_xray_pneu.pdf', dpi=1024)

# Show the plot
plt.show()


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from skimage import img_as_ubyte

# Set the figure size
fig = plt.figure(figsize=(8, 128))  # Adjust the overall figure size

# Number of random images to select
num_images_to_plot = 64

# Generate random indices for selecting images
random_indices = random.sample(range(len(images_test)), num_images_to_plot)

# Create a GridSpec with three columns (for the original, denoised, and adversarial images)
# Increase width_ratios to make each subplot larger
gs = GridSpec(num_images_to_plot, 3, width_ratios=[1, 1, 1])

for i, index in enumerate(random_indices):
    # Original image
    original_img = images_test[index]
    original_prediction_label = labels_test[index]

    # Adversarial image 
    adversarial_img_pgd = adversarial_x_pgd[index]
    pred_x_pgd1 = np.argmax(pred_x_pgd, axis=1)
    pred_x_pgd2 = pred_x_pgd1[index]

    # Denoised image
    denoised_img_pgd = denoised_x_pgd[index]
    denoised_pred_x_pgd1 = np.argmax(denoised_pred_x_pgd, axis=1)
    denoised_pred_x_pgd2 = denoised_pred_x_pgd1[index]

    # Create subplots within the GridSpec with aspect ratio matching the images
    ax0 = plt.subplot(gs[i, 0])
    ax1 = plt.subplot(gs[i, 1])
    ax2 = plt.subplot(gs[i, 2])

    # Plot the original image on the left
    ax0.imshow(original_img, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax0.set_title(f'Original\nPrediction: {original_prediction_label}') 

    # Plot the adversarial image in the middle
    ax1.imshow(adversarial_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax1.set_title(f'PGD\nPrediction: {pred_x_pgd2}') 

    # Plot the denoised image on the right
    ax2.imshow(denoised_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax2.set_title(f'Purified\nPrediction: {denoised_pred_x_pgd2}') 

# Adjust subplot spacing
plt.tight_layout()

# Save the plot with reduced DPI as JPEG file
#plt.savefig('qualitative_analysis_images_chest_xray_pneu_pgd2.jpg', dpi=1024)
plt.savefig('qualitative_analysis_images_chest_xray_pneu_2.pdf', dpi=1024)

# Show the plot
plt.show()

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from skimage import img_as_ubyte

# Set the figure size
fig = plt.figure(figsize=(8, 128))  # Adjust the overall figure size

# Number of random images to select
num_images_to_plot = 64

# Generate random indices for selecting images
random_indices = random.sample(range(len(images_test)), num_images_to_plot)

# Create a GridSpec with three columns (for the original, denoised, and adversarial images)
# Increase width_ratios to make each subplot larger
gs = GridSpec(num_images_to_plot, 3, width_ratios=[1, 1, 1])

for i, index in enumerate(random_indices):
    # Original image
    original_img = images_test[index]
    original_prediction_label = labels_test[index]

    # Adversarial image 
    adversarial_img_pgd = adversarial_x_bim[index]
    pred_x_pgd1 = np.argmax(pred_x_bim, axis=1)
    pred_x_pgd2 = pred_x_pgd1[index]

    # Denoised image
    denoised_img_pgd = denoised_x_bim[index]
    denoised_pred_x_pgd1 = np.argmax(denoised_pred_x_bim, axis=1)
    denoised_pred_x_pgd2 = denoised_pred_x_pgd1[index]

    # Create subplots within the GridSpec with aspect ratio matching the images
    ax0 = plt.subplot(gs[i, 0])
    ax1 = plt.subplot(gs[i, 1])
    ax2 = plt.subplot(gs[i, 2])

    # Plot the original image on the left
    ax0.imshow(original_img, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax0.set_title(f'Original\nPrediction: {original_prediction_label}') 

    # Plot the adversarial image in the middle
    ax1.imshow(adversarial_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax1.set_title(f'BIM\nPrediction: {pred_x_pgd2}') 

    # Plot the denoised image on the right
    ax2.imshow(denoised_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax2.set_title(f'Purified\nPrediction: {denoised_pred_x_pgd2}') 

# Adjust subplot spacing
plt.tight_layout()

# Save the plot with reduced DPI as JPEG file
#plt.savefig('chest_xray_pneu_bim1.jpg', dpi=1024)
plt.savefig('chest_xray_pneu_bim1.pdf', dpi=1024)

# Show the plot
plt.show()

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from skimage import img_as_ubyte

# Set the figure size
fig = plt.figure(figsize=(8, 128))  # Adjust the overall figure size

# Number of random images to select
num_images_to_plot = 64

# Generate random indices for selecting images
random_indices = random.sample(range(len(images_test)), num_images_to_plot)

# Create a GridSpec with three columns (for the original, denoised, and adversarial images)
# Increase width_ratios to make each subplot larger
gs = GridSpec(num_images_to_plot, 3, width_ratios=[1, 1, 1])

for i, index in enumerate(random_indices):
    # Original image
    original_img = images_test[index]
    original_prediction_label = labels_test[index]

    # Adversarial image 
    adversarial_img_pgd = adversarial_x_mim[index]
    pred_x_pgd1 = np.argmax(pred_x_mim, axis=1)
    pred_x_pgd2 = pred_x_pgd1[index]

    # Denoised image
    denoised_img_pgd = denoised_x_mim[index]
    denoised_pred_x_pgd1 = np.argmax(denoised_pred_x_mim, axis=1)
    denoised_pred_x_pgd2 = denoised_pred_x_pgd1[index]

    # Create subplots within the GridSpec with aspect ratio matching the images
    ax0 = plt.subplot(gs[i, 0])
    ax1 = plt.subplot(gs[i, 1])
    ax2 = plt.subplot(gs[i, 2])

    # Plot the original image on the left
    ax0.imshow(original_img, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax0.set_title(f'Original\nPrediction: {original_prediction_label}') 

    # Plot the adversarial image in the middle
    ax1.imshow(adversarial_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax1.set_title(f'MIM\nPrediction: {pred_x_pgd2}') 

    # Plot the denoised image on the right
    ax2.imshow(denoised_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax2.set_title(f'Purified\nPrediction: {denoised_pred_x_pgd2}') 

# Adjust subplot spacing
plt.tight_layout()

# Save the plot with reduced DPI as JPEG file
plt.savefig('mim3.pdf', dpi=1024)

# Show the plot
plt.show()

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from skimage import img_as_ubyte

# Set the figure size
fig = plt.figure(figsize=(8, 8))  # Adjust the overall figure size

# Number of random images to select
num_images_to_plot = 4

# Generate random indices for selecting images
random_indices = random.sample(range(len(images_test)), num_images_to_plot)

# Create a GridSpec with three columns (for the original, denoised, and adversarial images)
# Increase width_ratios to make each subplot larger
gs = GridSpec(num_images_to_plot, 3, width_ratios=[1, 1, 1])

for i, index in enumerate(random_indices):
    # Original image
    original_img = images_test[index]
    original_prediction_label = labels_test[index]

    # Adversarial image 
    adversarial_img_pgd = adversarial_x_fgsm[index]
    pred_x_pgd1 = np.argmax(pred_x_fgsm, axis=1)
    pred_x_pgd2 = pred_x_pgd1[index]

    # Denoised image
    denoised_img_pgd = denoised_x_fgsm[index]
    denoised_pred_x_pgd1 = np.argmax(denoised_pred_x_fgsm, axis=1)
    denoised_pred_x_pgd2 = denoised_pred_x_pgd1[index]

    # Create subplots within the GridSpec with aspect ratio matching the images
    ax0 = plt.subplot(gs[i, 0])
    ax1 = plt.subplot(gs[i, 1])
    ax2 = plt.subplot(gs[i, 2])

    # Plot the original image on the left
    ax0.imshow(original_img, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax0.set_title(f'Original\nPrediction: {original_prediction_label}') 

    # Plot the adversarial image in the middle
    ax1.imshow(adversarial_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax1.set_title(f'FGSM\nPrediction: {pred_x_pgd2}') 

    # Plot the denoised image on the right
    ax2.imshow(denoised_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax2.set_title(f'Purified\nPrediction: {denoised_pred_x_pgd2}') 

# Adjust subplot spacing
plt.tight_layout()

# Save the plot with reduced DPI as JPEG file
plt.savefig('qualitative_analysis_images_chest_xray_pneu_fgsm_4.jpg', dpi=1024)

# Show the plot
plt.show()

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from skimage import img_as_ubyte

# Set the figure size
fig = plt.figure(figsize=(10, 10))  # Increase the overall figure size

# Number of random images to select
num_images_to_plot = 4

# Generate random indices for selecting images
random_indices = random.sample(range(len(images_test)), num_images_to_plot)

# Create a GridSpec with three columns (for the original, denoised, and adversarial images)
# Increase width_ratios to make each subplot larger
gs = GridSpec(num_images_to_plot, 3, width_ratios=[1, 1, 1])

for i, index in enumerate(random_indices):
    # Original image
    original_img = images_test[index]
    original_prediction_label = labels_test[index]

    # Adversarial image 
    adversarial_img_pgd = adversarial_x_pgd[index]
    pred_x_pgd1 = np.argmax(pred_x_pgd, axis=1)
    pred_x_pgd2 = pred_x_pgd1[index]

    # Denoised image
    denoised_img_pgd = denoised_x_pgd[index]
    denoised_pred_x_pgd1 = np.argmax(denoised_pred_x_pgd, axis=1)
    denoised_pred_x_pgd2 = denoised_pred_x_pgd1[index]

    # Create subplots within the GridSpec with aspect ratio matching the images
    ax0 = plt.subplot(gs[i, 0])
    ax1 = plt.subplot(gs[i, 1])
    ax2 = plt.subplot(gs[i, 2])

    # Plot the original image on the left
    ax0.imshow(original_img, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax0.set_title(f'Original\nPrediction: {original_prediction_label}') 

    # Plot the adversarial image in the middle
    ax1.imshow(adversarial_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax1.set_title(f'PGD\nPrediction: {pred_x_pgd2}') 

    # Plot the denoised image on the right
    ax2.imshow(denoised_img_pgd, cmap='gray')  # Assuming grayscale images, change cmap as needed
    ax2.set_title(f'Purified\nPrediction: {denoised_pred_x_pgd2}') 

# Adjust subplot spacing
plt.tight_layout()
plt.savefig('qualitative_analysis_images_chest_xray_pneu_1.jpg', dpi=1024)

# Show the plot
plt.show()


In [ ]:
def samples(G, noize, labels):
        images1 = G.predict([noize, labels])
        ys = np.argmax(labels, axis = 1)
        plt.figure(figsize = (12, 4))
        for i in range(16):
            plt.subplot(2, 8, (i + 1))
            plt.imshow(images1[i], cmap = 'gray')
            plt.title(ys[i])
        plt.show()
samples(G, latent_code, one_hot_labels)

In [ ]:
latent_code1 = tf.Variable(tf.random.normal(shape=(1, 128)))  # Initialize with random values

# Use mixed precision for the optimizer
optimizer = tf.keras.mixed_precision.LossScaleOptimizer(tf.keras.optimizers.Adam(learning_rate=0.01))

#reshaped_tensor = tf.reduce_mean(images, axis=(2, 3))

gradient_accumulation_steps = 4

for i in range(100):
    with tf.GradientTape() as tape:
        generated_image = G([latent_code1, one_hot_labels])
        reconstruction_loss = tf.reduce_mean(tf.square(generated_image - tf.stop_gradient(images)))

    gradients = tape.gradient(reconstruction_loss, [latent_code] + G.trainable_variables)
    if (i + 1) % gradient_accumulation_steps == 0:
        optimizer.apply_gradients(zip(gradients, [latent_code] + G.trainable_variables))

# Assuming one_hot_encoded is generated based on your labels
denoised_image = G([tf.stop_gradient(latent_code), one_hot_labels])
print("Denoised Image Shape:", denoised_image.shape)

In [ ]:
# Access the second last dense layer by name

#optimizer = tf.optimizers.Adam(learning_rate=0.01)
# Example:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
#optimizer = tf.keras.mixed_precision.experimental.LossScaleOptimizer(optimizer, "dynamic")
optimizer = tf.keras.mixed_precision.LossScaleOptimizer(tf.keras.optimizers.Adam(learning_rate=0.01))

import pandas as pd
reshaped_tensor = tf.reduce_mean(images, axis=(2, 3))
print(reshaped_tensor.shape)

gradient_accumulation_steps = 4

for i in range(100):
    with tf.GradientTape() as tape:
        generated_image = G([reshaped_tensor, one_hot_labels])
        reconstruction_loss = tf.reduce_mean(tf.square(generated_image - tf.stop_gradient(images)))

    gradients = tape.gradient(reconstruction_loss, [latent_code] + G.trainable_variables)
    if (i + 1) % gradient_accumulation_steps == 0:
        optimizer.apply_gradients(zip(gradients, [latent_code] + G.trainable_variables))
        
# Assuming one_hot_encoded is generated based on your labels
denoised_image = G.predict([tf.stop_gradient(latent_code), one_hot_encoded])
print("Denoised Image Shape:", denoised_image.shape)

In [ ]:
resnet18_model.evaluate(denoised_images_pgd, one_hot_labels)

In [ ]:
G = tf.keras.models.load_model('/working/generator.h5')

In [ ]:
datasetGenerationSize = 30000
noize = tf.random.uniform(shape = (datasetGenerationSize, 100), minval = -1, maxval = 1)
newlabels = tf.keras.utils.to_categorical(np.random.choice([0, 1], size = (datasetGenerationSize, )), num_classes = 2)

In [ ]:
noize.shape, newlabels.shape

In [ ]:
np.unique(np.argmax(newlabels, axis = 1), return_counts = True)

In [ ]:
imagesGeneration = G.predict([noize, newlabels])
imagesGeneration.shape

- Samples generated by the generator for each case (healthy person, person with pneumonia).

In [ ]:
plt.figure(figsize = (12, 12))
t = np.argmax(newlabels, axis = 1)
for i in range(64):
    plt.subplot(8, 8, (i + 1))
    plt.imshow(imagesGeneration[i])
    plt.title(t[i])
plt.legend()

In order to be able to evaluate the images generated by the generating neural network, we can do so by proposing a neural structure dedicated to classifying the images generated by the generating neural network, and then we return to the basic images included in the dataset, and we evaluate the performance of the classified neural network that It was trained on the generated images, in order to see if the learned characteristics of the generated images can give high results on the basic images included in the dataset.

In [ ]:
basemodel = tf.keras.applications.VGG16(weights = None, input_shape = (64, 64, 3),
                                        pooling = 'max', include_top = False)
x = layers.Dropout(0.4)(basemodel.output)
x = layers.Dense(128,)(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU(alpha = 0.2)(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(32,)(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU(alpha = 0.2)(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(1, activation = 'sigmoid')(x)
m = tf.keras.models.Model(inputs = basemodel.input, outputs = x)
m.compile(loss = 'binary_crossentropy', optimizer = tf.keras.optimizers.Adam(learning_rate = 0.00001))
m.summary()

- Training the neural network to classify the images generated by the generator.

In [ ]:
history = m.fit(imagesGeneration, np.argmax(newlabels, axis = 1),
                epochs = 60, batch_size = 64,
                validation_split = 0.2,
                callbacks = [tf.keras.callbacks.EarlyStopping(patience = 2, monitor = 'val_loss', mode = 'min',
                                                              restore_best_weights = True)])

In [ ]:
plt.figure(figsize = (7, 6))
plt.plot(history.history['loss'], label = 'training loss')
plt.plot(history.history['val_loss'], label = 'validation loss')
plt.title('Results obtained while training a neural network on images generated by the neural network')
plt.legend()

- Now, after training on the images generated by the generator, we will test the neural network on the basic images included in the dataset.

- We will use several measures in the evaluation to study what is the ability of the generative adversarial network to capture the basic features that characterize each class, and whether the second classified network extracted the features included in the generated images.
- Are the attributes that were extracted from the images generated by the generator, can be used on the original images included in the dataset.
- This helps in the ability to study what was actually generated, and whether the focus was really on the cases that the X-ray images made him have pneumonia or not.

In [ ]:
m.evaluate(images, labels)

In [ ]:
y_pred = tf.squeeze(m.predict(images))
y_pred.shape

In [ ]:
y_pred = y_pred >= 0.5
y_pred = np.array(y_pred, dtype = 'int32')
y_pred

In [ ]:
accuracy_score(y_pred, labels)*100

In [ ]:
print(classification_report(y_pred, labels))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
cm = confusion_matrix(y_pred, labels)
cm

In [ ]:
import pandas as pd
cmObject = pd.DataFrame(cm , index = ['NORMAL', 'PNEUMONIA'],
                        columns = ['NORMAL', 'PNEUMONIA'])
cmObject.head()

In [ ]:
print('f1_score: {}, recall_score: {}, precision_score: {}'.format(f1_score(y_pred, labels)*100,
                                                                   recall_score(y_pred, labels)*100,
                                                                   precision_score(y_pred, labels)*100))

In [ ]:
sns.heatmap(cmObject, annot = True, cmap="Blues")

- In the end, we can see that we have reached a neural network that is able to generate accurate images.
But the slight variation in the accuracy of the classification for each class is due to the fact that we need more training time for the generative adversarial network, which helps to focus more on the characteristics of each class (because the number of samples in the basic dataset is different for each class (the healthy case, pneumonia)).